# Análisis de Fallas HVDC — STN 25 Barras (v1.0)
## Corrección del cálculo monofásico sobre corredores de línea

Correcciones incorporadas respecto de las iteraciones previas de desarrollo:

1. **Z_barra de secuencia cero construida desde `DATA0`.** Anteriormente `DATA0` se
   importaba pero nunca se usaba: `calibrar_zbarra_sim` copiaba la matriz de
   secuencia positiva en ambas secuencias.
2. **Filtro de cobertura en `tension_falla_linea_1f`.** El cálculo monofásico
   sobre un corredor exige `Z0_mk` simulado en ambos extremos. Interpolar entre
   un extremo calibrado y otro algorítmico mezcla escalas separadas por dos
   órdenes de magnitud y producía perfiles imposibles (mínimos interiores de
   ~0 p.u. en corredores cuyos dos extremos superaban 0,94 p.u.).
3. **El clamp a [0,1] deja de ser silencioso**: emite un aviso por corredor.
4. **Barra 6 renombrada** a Punta Colorada.
5. **Rótulo corregido**: la columna monofásica es `|V_a,mk|` (fase fallada),
   no `|V(1)_mk|` (secuencia positiva).

Resultados esperados: **4,3995 / 5,8817 / 4,3890 FC/año**
(Cardones / Alto Jahuel / Valdivia).

> **Pendiente.** No existe dato calibrado de `Z0_kj`; el código mantiene
> `Z0_kj = Z1_kj`, dimensionalmente incorrecto pero de escala compatible.
> Sustituirlo por el valor algorítmico empeora el resultado. La solución
> definitiva requiere una simulación monofásica por corredor:
> `Z0_kj = (1 - V_a,kj_sim)·(2·Z1_jj + Z0_jj) - 2·Z1_kj`


## Módulo 1 — `sic_datos.py`

In [ ]:
%%writefile sic_datos.py
"""
sic_datos.py — Parámetros del STN simplificado de 25 barras  (v1.0)
====================================================================
Fuente: VR (2026), Cuadros 4.1, 4.2 y 5.1

v1.0: corrige el nombre de la barra 6 (Punta Colorada, no Colorado).

Bases: Sbase=100 MVA, Vbase_220=220 kV (Zbase=484 Ω), Vbase_500=500 kV (Zbase=2500 Ω)
"""
import numpy as np

SBASE     = 100.0
VBASE_220 = 220.0
VBASE_500 = 500.0
ZBASE_220 = VBASE_220**2 / SBASE   # 484 Ω
ZBASE_500 = VBASE_500**2 / SBASE   # 2500 Ω
NBUS      = 25
V_UMBRAL  = 0.9  # p.u. — umbral área de vulnerabilidad (V_mp < V_UMBRAL ⇒ crítico)

# Placeholder para pares (m,k) sin dato de simulación: límite inferior conocido
# que garantiza clasificación "no crítico" en calibrar_zbarra_sim y en las
# funciones de tensiones_falla.py. Con el criterio estrictamente "<" (no "≤"),
# coincidir exactamente con V_UMBRAL ya es seguro (0.9 < 0.9 es False), así
# que se reutiliza el mismo valor — no se necesita un margen artificial.
V_NO_CRITICO = V_UMBRAL  # p.u.

# Modelos de probabilidad de FC (Ecs. 4.1 y 4.2)
V_UMBRAL_3F   = 0.85   # umbral escalón falla trifásica (Ec. 4.1 / ec:pfc_trifasica)
V_TRANS_1F_HI = 0.9    # inicio transición lineal falla monofásica (P_FC=0) (Ec. 4.2 / ec:pfc_monofasica)
V_TRANS_1F_LO = 0.8    # fin transición (P_FC=1) falla monofásica (Ec. 4.2 / ec:pfc_monofasica)

NOMBRES = {
    1:'Paposo 220',        2:'D. de Almagro 220',  3:'Carrera Pinto 220',
    4:'Cardones 220',      5:'Maitencillo 220',     6:'Punta Colorada 220',
    7:'Pan de Azúcar 220', 8:'Los Vilos 220',       9:'Nogales 220',
    10:'Quillota 220',     11:'Polpaico 220',        12:'Polpaico 500',
    13:'Cerro Navia 220',  14:'Alto Jahuel 220',     15:'Alto Jahuel 500',
    16:'Ancoa 500',        17:'Ancoa 220',           18:'Itahue 220',
    19:'Charrúa 500',      20:'Charrúa 220',         21:'Concepción 220',
    22:'Temuco 220',       23:'Cautín 220',          24:'Valdivia 220',
    25:'Puerto Montt 220',
}

BARRAS_OBS = {4:'Cardones 220', 15:'Alto Jahuel 500', 24:'Valdivia 220'}

# ===========================================================================
# Cuadro 5.1: Potencias de cortocircuito de simulación DIgSILENT
# Listas indexadas 0..24 → barra 1..25
# ===========================================================================
SCC3_MVA = [
    367.10,  510.77,  721.62, 1292.37, 2280.97,
   1600.03, 1538.95, 2089.34, 4638.67, 8274.73,
   7949.67, 6871.32, 5711.92, 6111.86, 6970.92,
   7366.94, 4725.33, 2829.57, 7033.78, 7606.27,
   2339.57, 2287.57, 2319.55, 1464.26, 1080.99,
]

SCC1_MVA = [
    173.00,  218.21,  249.28,  563.32,  869.98,
    465.30,  587.35,  634.04, 1019.21, 3165.93,
   2269.14, 1992.09, 1829.71, 2531.19, 2444.07,
   2090.59, 1510.24, 1134.07, 2319.66, 3052.15,
   1068.92,  770.09,  770.03,  577.36,  427.46,
]


# ===========================================================================
# Tensiones de falla simuladas — DIgSILENT (Cuadro 5.2)
# Clave (m, k): tensión en barra de observación m ante falla en barra k.
# Solo se incluyen valores ≤ 0.90 p.u. (los que producen riesgo de falla
# de conmutación). Las entradas ausentes implican tensión > 0.90 p.u.
# ===========================================================================

V_SIM_3F = {
    # ── Cardones 220 (bus 4) ─────────────────────────────────────────────
    (4,  1): 0.787, (4,  2): 0.670, (4,  3): 0.488, (4,  4): 0.000,
    (4,  5): 0.003, (4,  6): 0.502, (4,  7): 0.639, (4,  8): 0.853,
    (4,  9): 0.895,
    # Nota: el par (4,15) NO tiene dato real de simulación, pero ya no
    # requiere un valor explícito aquí — calibrar_zbarra_sim (zbarra.py)
    # lo resuelve mediante la regla simétrica (mínimo de Z1kk_sim entre
    # ambas barras de observación), que garantiza V≥V_NO_CRITICO en ambas
    # direcciones sin depender del orden de iteración. Ver zbarra.py.
    # ── Alto Jahuel 500 (bus 15) ─────────────────────────────────────────
    (15, 10): 0.817, (15, 11): 0.630, (15, 12): 0.227,
    (15, 13): 0.832, (15, 14): 0.347, (15, 15): 0.000,
    (15, 16): 0.283, (15, 17): 0.729,
    (15, 19): 0.470, (15, 20): 0.638,
    # ── Valdivia 220 (bus 24) ────────────────────────────────────────────
    (24, 12): 0.873, (24, 15): 0.866, (24, 16): 0.758,
    (24, 19): 0.685, (24, 20): 0.476, (24, 22): 0.331,
    (24, 23): 0.315, (24, 24): 0.000, (24, 25): 0.570,
}

# ===========================================================================
# Impedancias de transferencia simuladas — DIgSILENT
# Clave (m, k): impedancia de transferencia Z_mk (en p.u.) entre barra
# de observación m y barra de falla k.
#
# Z_MK_1 — secuencia positiva (= secuencia negativa, red pasiva)
#   Fuente: mismos datos de V_SIM_3F, pero expresados directamente como
#   impedancia: Z1_mk = (1 - V_sim_3f) * Z1_kk.
#   Usados en tension_falla_barra_1f y tension_falla_linea_1f como
#   numerador Z1_mp = (1-xi)*Z1_mk + xi*Z1_mj (igual que en rama 3f).
#
# Z_MK_0 — secuencia cero
#   Fuente: simulación directa en PowerFactory (informe de secuencias).
#   Algunos valores son negativos — físicamente posible en elementos
#   fuera de la diagonal de Z0_barra cuando transformadores Dyn bloquean
#   la secuencia cero e invierten el sentido de la transferencia.
#   Usados en la fórmula exacta de tensión de fase:
#     V_a_mk = 1 - (2*Z1_mk + Z0_mk) / (2*Z1_kk + Z0_kk)
#   reemplazando la derivación analítica vía Ec. 3.40 (que usaba solo
#   secuencia positiva) y el parche Z1_eff_1f de v1.0 (que absorbía
#   el efecto de Z0_mk en un Z1_mk ficticio distinto al de la rama 3f).
# ===========================================================================

Z_MK_1 = {
    # ── Cardones 220 (bus 4) ─────────────────────────────────────────────
    (4,  1): 0.0580, (4,  2): 0.0646, (4,  3): 0.0710, (4,  4): 0.0774,
    (4,  5): 0.0437, (4,  6): 0.0311, (4,  7): 0.0235, (4,  8): 0.0070,
    (4,  9): 0.0022, (4, 10): 0.0012,
    # ── Alto Jahuel 500 (bus 15) ─────────────────────────────────────────
    (15,  9): 0.004752, (15, 10): 0.0022, (15, 11): 0.0047,
    (15, 12): 0.0113,  (15, 13): 0.0029, (15, 14): 0.0105,
    (15, 15): 0.0143,  (15, 16): 0.0098, (15, 17): 0.0057,
    (15, 18): 0.0033,  (15, 19): 0.0075, (15, 20): 0.0047,
    (15, 21): 0.00047, (15, 22): 0.0,    (15, 23): 0.0,
    # ── Valdivia 220 (bus 24) ────────────────────────────────────────────
    (24, 12): 0.0019, (24, 15): 0.0019, (24, 16): 0.0033,
    (24, 19): 0.0045, (24, 20): 0.0069, (24, 21): 0.0023,
    (24, 22): 0.0292, (24, 23): 0.0295, (24, 24): 0.0683, (24, 25): 0.0398,
}

Z_MK_0 = {
    # ── Cardones 220 (bus 4) ─────────────────────────────────────────────
    (4,  1): -0.0038, (4,  2): -0.0024, (4,  3):  0.0021, (4,  4):  0.0227,
    (4,  5):  0.0013, (4,  6): -0.0083, (4,  7): -0.0031, (4,  8): -0.0059,
    # ── Alto Jahuel 500 (bus 15) ─────────────────────────────────────────
    (15,  9): -0.003398, (15, 10): -0.000873, (15, 11):  0.0016,
    (15, 12):  0.0047,   (15, 13): -0.0011,   (15, 14):  0.0067,
    (15, 15):  0.0123,   (15, 16):  0.0014,   (15, 17):  0.00082,
    (15, 18): -0.0011,   (15, 19): -0.000074, (15, 20): -0.000092,
    (15, 21): -0.0008,   (15, 22): -0.00449,  (15, 23): -0.00451,
    # ── Valdivia 220 (bus 24) ────────────────────────────────────────────
    (24, 16): -0.0025, (24, 19): -0.0016, (24, 20): -0.00065,
    (24, 21): -0.0013, (24, 22):  0.0014, (24, 23):  0.0016,
    (24, 24):  0.0366, (24, 25):  0.0012,
}

# V_SIM_1F — tensiones de fase real simuladas (PowerFactory), derivadas
# analíticamente de Z_MK_1 y Z_MK_0 mediante la fórmula exacta:
#   V_a_mk = 1 - (2*Z1_mk + Z0_mk) / (2*Z1_kk + Z0_kk)
# Se mantiene como referencia de validación y para los pares sin Z0_mk.
# Solo se incluyen valores ≤ 0.90 p.u.
V_SIM_1F = {
    # ── Cardones 220 (bus 4) ─────────────────────────────────────────────
    (4,  1): 0.806, (4,  2): 0.723, (4,  3): 0.641, (4,  4): 0.000,
    (4,  5): 0.228, (4,  6): 0.750, (4,  7): 0.742,
    # ── Alto Jahuel 500 (bus 15) ─────────────────────────────────────────
    (15, 10): 0.888, (15, 11): 0.749, (15, 12): 0.456,
    (15, 14): 0.298, (15, 15): 0.000, (15, 16): 0.560,
    (15, 17): 0.815, (15, 19): 0.654, (15, 20): 0.711,
    # ── Valdivia 220 (bus 24) ────────────────────────────────────────────
    (24, 19): 0.821, (24, 20): 0.594, (24, 22): 0.538,
    (24, 23): 0.532, (24, 24): 0.000, (24, 25): 0.654,
}


# ===========================================================================
# Tensiones V_kj — tensión en barra k ante falla en barra j (extremos línea)
# Permite calcular Z_kj = (1-V_kj) × Z_jj  (simetría: Z_kj = Z_jk)
# ===========================================================================
V_KJ = {
    (1,  2): 0.0000, (2,  3): 0.0000, (3,  4): 0.0827, (4,  5): 0.0000,
    (5,  6): 0.5040, (6,  7): 0.2750, (7,  8): 0.5090, (8,  9): 0.1870,
    (9, 10): 0.0500, (10,11): 0.4290, (11,13): 0.4690, (13,14): 0.7160,
    (12,15): 0.2098, (12,16): 0.3010, (15,16): 0.2794, (16,19): 0.2520,
    (17,18): 0.5710, (20,21): 0.8280, (20,22): 0.8560, (20,23): 0.8520,
    (23,24): 0.5681, (24,25): 0.5697,
}


def get_zkj_simulacion(Z1kk_sim: list) -> dict:
    """
    Calcula Z_kj para cada par extremo de línea desde tensiones simuladas.
    Z_kj = (1 - V_kj) × Z_jj_sim  (Ec. 3.8 inversa, simétrica)
    Retorna dict {(k,j): Z_kj_pu} con claves en ambos sentidos.
    """
    zkj = {}
    for (k, j), vkj in V_KJ.items():
        z = (1.0 - vkj) * Z1kk_sim[j - 1]
        zkj[(k, j)] = z
        zkj[(j, k)] = z
    return zkj


def get_zkk_simulacion():
    """
    Z1_kk = Sbase / Scc3        (Ec. 3.33 inversa)
    Z0_kk = Sbase/Scc1 - 2·Z1  (Ec. 3.34 inversa)
    Datos exactos para las 25 barras — sin aproximaciones.
    """
    Z1kk = [SBASE / s for s in SCC3_MVA]
    Z0kk = [SBASE / SCC1_MVA[k] - 2*Z1kk[k] for k in range(NBUS)]
    return Z1kk, Z0kk


# ===========================================================================
# Cuadro 4.1: líneas de transmisión
# (fb, tb, X1[Ω/km], X0[Ω/km], L[km], V[kV])  — un circuito por fila
# ===========================================================================
_LINEAS = [
    (1,  2, 0.3998,1.4839,185.00,220),  #  0  Paposo – D. Almagro       c1
    (1,  2, 0.3998,1.4839,185.00,220),  #  1                             c2
    (2,  3, 0.3932,1.3088, 72.15,220),  #  2  D. Almagro – C. Pinto
    (3,  4, 0.3977,1.3227, 75.30,220),  #  3  Carrera Pinto – Cardones
    (4,  5, 0.4064,1.3227,132.70,220),  #  4  Cardones – Maitencillo     c1
    (4,  5, 0.4064,1.3227,133.30,220),  #  5                             c2
    (4,  5, 0.4064,1.3227,133.30,220),  #  6                             c3
    (5,  6, 0.3912,1.3114,112.60,220),  #  7  Maitencillo – P. Colorado  c1
    (5,  6, 0.3912,1.3114,112.60,220),  #  8                             c2
    (6,  7, 0.3912,1.3114, 84.00,220),  #  9  P. Colorado – Pan Azúcar   c1
    (6,  7, 0.3912,1.3114, 84.00,220),  # 10                             c2
    (7,  8, 0.3900,1.3114,228.00,220),  # 11  Pan Azúcar – Los Vilos     c1
    (7,  8, 0.3900,1.3114,228.00,220),  # 12                             c2
    (8,  9, 0.3938,1.3159, 97.10,220),  # 13  Los Vilos – Nogales        c1
    (8,  9, 0.3938,1.3159, 97.10,220),  # 14                             c2
    (9, 10, 0.3938,1.3159, 27.00,220),  # 15  Nogales – Quillota         c1
    (9, 10, 0.3938,1.3159, 27.00,220),  # 16                             c2
    (10,11, 0.2370,1.1185, 49.58,220),  # 17  Quillota – Polpaico 220    c1
    (10,11, 0.2370,1.1185, 49.58,220),  # 18                             c2
    (11,13, 0.4044,1.3903, 29.80,220),  # 19  Polpaico – Cerro Navia     c1
    (11,13, 0.4044,1.3903, 29.80,220),  # 20                             c2
    (13,14, 0.3496,1.3253, 39.20,220),  # 21  Cerro Navia – AJ 220       c1
    (13,14, 0.3496,1.3253, 39.20,220),  # 22                             c2
    (12,15, 0.2745,1.2314, 71.89,500),  # 23  Polpaico 500 – AJ 500
    (12,16, 0.2774,1.0653,309.59,500),  # 24  Polpaico 500 – Ancoa 500 (directo)
    (15,16, 0.3351,1.0404,240.49,500),  # 25  AJ 500 – Ancoa 500 (NUEVO)
    (16,19, 0.3339,1.0714,182.83,500),  # 26  Ancoa 500 – Charrúa 500    c1
    (16,19, 0.3310,1.0690,196.14,500),  # 27                             c2
    (17,18, 0.3840,1.2052, 65.00,220),  # 28  Ancoa 220 – Itahue         1 circuito
    (20,21, 0.3869,1.3435, 71.80,220),  # 29  Charrúa 220 – Concepción
    (20,22, 0.3955,1.3558,195.70,220),  # 30  Charrúa 220 – Temuco
    (20,23, 0.2900,1.2920,204.00,220),  # 31  Charrúa 220 – Cautín       c1
    (20,23, 0.2900,1.2920,204.00,220),  # 32                             c2
    (22,23, 0.3975,1.3711,  3.00,220),  # 33  Temuco – Cautín            c1
    (22,23, 0.3975,1.3711,  3.00,220),  # 34                             c2
    (23,24, 0.4058,1.3795,149.12,220),  # 35  Cautín – Valdivia          c1
    (23,24, 0.4058,1.3795,149.12,220),  # 36                             c2
    (24,25, 0.3978,1.3703,207.04,220),  # 37  Valdivia – Puerto Montt    c1
    (24,25, 0.3978,1.3703,207.04,220),  # 38                             c2
]

# Cuadro 4.2: transformadores (fb, tb, X1_pu, X0_pu)
_XFMR = [
    (11,12,0.0218,0.0148),(14,15,0.0216,0.0147),
    (16,17,0.0218,0.0148),(19,20,0.0202,0.0202),
]


def _xpu(x_ohm_km, L_km, V_kv):
    zb = ZBASE_220 if V_kv == 220 else ZBASE_500
    return x_ohm_km * L_km / zb


def _make_data():
    lpu = [(fb,tb,_xpu(x1,L,V),_xpu(x0,L,V)) for (fb,tb,x1,x0,L,V) in _LINEAS]
    xpu = list(_XFMR)

    # Árbol generador (id=1) — añade cada barra exactamente una vez
    # Índices referenciados a la nueva _LINEAS (39 entradas):
    #   0:1-2c1  2:2-3  3:3-4  4:4-5c1  7:5-6c1  9:6-7c1  11:7-8c1
    #  13:8-9c1  15:9-10c1  17:10-11c1  19:11-13c1  21:13-14c1
    #  xpu[0]:11→12  23:12→15  24:12→16  xpu[2]:16→17  28:17→18
    #  26:16→19c1  xpu[3]:19→20  29:20→21  30:20→22  31:20→23c1
    #  35:23→24c1  37:24→25c1
    tree = [
        lpu[0],  lpu[2],  lpu[3],  lpu[4],  lpu[7],  lpu[9],  lpu[11], lpu[13],
        lpu[15], lpu[17], lpu[19], lpu[21], xpu[0],  lpu[23], lpu[24],
        xpu[2],  lpu[28], lpu[26], xpu[3],  lpu[29], lpu[30], lpu[31],
        lpu[35], lpu[37],
    ]

    # Mallas (id=-1) — circuitos paralelos y enlace de cierre 15-16 (nuevo)
    #   1:1-2c2  5:4-5c2  6:4-5c3  8:5-6c2  10:6-7c2  12:7-8c2  14:8-9c2
    #  16:9-10c2  18:10-11c2  20:11-13c2  22:13-14c2  xpu[1]:14-15
    #  25:15-16(NUEVO)  27:16-19c2  32:20-23c2  33:22-23c1  34:22-23c2
    #  36:23-24c2  38:24-25c2
    mesh = [
        lpu[1],  lpu[5],  lpu[6],  lpu[8],  lpu[10], lpu[12], lpu[14],
        lpu[16], lpu[18], lpu[20], lpu[22], xpu[1],  lpu[25],
        lpu[27], lpu[32], lpu[33], lpu[34], lpu[36], lpu[38],
    ]
    src = [(0,1,0.001,0.001)]
    data1, data0 = [], []
    for (fb,tb,x1,x0) in src:
        data1.append([fb,tb,x1, 0]); data0.append([fb,tb,x0, 0])
    for (fb,tb,x1,x0) in tree:
        data1.append([fb,tb,x1, 1]); data0.append([fb,tb,x0, 1])
    for (fb,tb,x1,x0) in mesh:
        data1.append([fb,tb,x1,-1]); data0.append([fb,tb,x0,-1])
    return np.array(data1,dtype=float), np.array(data0,dtype=float)


DATA1, DATA0 = _make_data()


def get_corredores():
    from collections import defaultdict
    grupos = defaultdict(list)
    for (fb,tb,x1,x0,L,V) in _LINEAS:
        key = (min(fb,tb), max(fb,tb))
        grupos[key].append((_xpu(x1,L,V), _xpu(x0,L,V), L))
    corredores = []
    for (fb,tb), ents in sorted(grupos.items()):
        x1_tot = 1.0 / sum(1.0/e[0] for e in ents)   # paralelo — para Z_barra
        x0_tot = 1.0 / sum(1.0/e[1] for e in ents)
        x1_circ = ents[0][0]   # un circuito individual — para z_kj serie en Ec. 3.16
        x0_circ = ents[0][1]
        corredores.append({
            'from': fb, 'to': tb,
            'label': f'{NOMBRES[fb]} — {NOMBRES[tb]}',
            'x1_pu':      x1_tot,   # equivalente paralelo
            'x0_pu':      x0_tot,
            'x1_pu_circ': x1_circ,  # circuito individual (z_kj serie)
            'x0_pu_circ': x0_circ,
            'longitud_km': ents[0][2]
        })
    return corredores


if __name__ == '__main__':
    Z1kk, Z0kk = get_zkk_simulacion()
    print(f"{'Bus':>4}  {'Nombre':<24}  {'Scc3[MVA]':>10}  "
          f"{'Scc1[MVA]':>10}  {'Z1kk':>8}  {'Z0kk':>8}")
    for k in range(NBUS):
        print(f"{k+1:>4}  {NOMBRES[k+1]:<24}  {SCC3_MVA[k]:>10.2f}  "
              f"{SCC1_MVA[k]:>10.2f}  {Z1kk[k]:>8.5f}  {Z0kk[k]:>8.5f}")
    print(f"\nCorredores: {len(get_corredores())}")


## Módulo 2 — `zbarra.py`

In [ ]:
%%writefile zbarra.py
"""
zbarra.py — Construcción de la Matriz de Impedancias de Barra
=============================================================
Implementa el algoritmo del Apéndice B de la memoria (VR, 2026),
basado en incorporación secuencial de elementos con Reducción de Kron.

Referencia de ecuaciones:
  Paso 1 (id=0): Ec. B.1  — rama al nodo de referencia
  Paso 2 (id=1): Ec. B.2  — nueva barra conectada a barra existente
  Paso 3 (id=-1): Ecs. B.3–B.5 — enlace entre barras existentes + Kron
"""

import numpy as np


def construir_zbarra(data: np.ndarray, nbus: int) -> np.ndarray:
    """
    Construye la matriz de impedancias de barra Z (nbus × nbus, compleja).

    Parámetros
    ----------
    data : ndarray shape (n, 4)
        Columnas: [barra_origen, barra_destino, X_pu, id]
        id =  0 → rama al nodo de referencia
        id =  1 → rama radial (agrega nueva barra)
        id = -1 → enlace de malla (Reducción de Kron)
    nbus : int
        Número total de barras del sistema.

    Retorna
    -------
    Z : ndarray shape (nbus, nbus), dtype complex
        Z[i, j] = impedancia entre barras (i+1) y (j+1).
        Los elementos diagonales Z[k,k] son las impedancias de Thévenin.
        Los elementos fuera de la diagonal Z[m,k] son las impedancias de
        transferencia usadas para calcular tensiones de falla.
    """
    fb  = data[:, 0].astype(int)
    tb  = data[:, 1].astype(int)
    x   = data[:, 2]              # reactancia en p.u. (real)
    ids = data[:, 3].astype(int)

    # Z_work: matriz que crece / se modifica durante el algoritmo
    Z_work = np.zeros((0, 0), dtype=complex)
    # Mapa barra_numero → índice en Z_work
    bus_idx: dict[int, int] = {}

    for k in range(len(fb)):
        xk  = 1j * x[k]       # impedancia pura imaginaria
        p   = fb[k]
        q   = tb[k]

        # ------------------------------------------------------------------
        # PASO 1: Rama al nodo de referencia (id=0) — Ec. B.1
        # ------------------------------------------------------------------
        if ids[k] == 0:
            idx_q = len(bus_idx)
            bus_idx[q] = idx_q
            n = len(Z_work)
            Z_new = np.zeros((n + 1, n + 1), dtype=complex)
            Z_new[:n, :n] = Z_work
            Z_new[n, n]   = xk
            Z_work = Z_new

        # ------------------------------------------------------------------
        # PASO 2: Nueva barra conectada a barra existente (id=1) — Ec. B.2
        # ------------------------------------------------------------------
        elif ids[k] == 1:
            if p not in bus_idx:
                raise ValueError(
                    f"Barra {p} no existe aún al procesar rama {p}→{q} (id=1). "
                    "Verificar orden del árbol en DATA."
                )
            p_i  = bus_idx[p]
            q_i  = len(bus_idx)
            bus_idx[q] = q_i
            n = len(Z_work)
            Z_new = np.zeros((n + 1, n + 1), dtype=complex)
            Z_new[:n, :n]  = Z_work
            Z_new[:n, n]   = Z_work[:, p_i]        # columna p copiada
            Z_new[n, :n]   = Z_work[p_i, :]        # fila p copiada (simetría)
            Z_new[n, n]    = Z_work[p_i, p_i] + xk # Zpp + zpq
            Z_work = Z_new

        # ------------------------------------------------------------------
        # PASO 3: Enlace de malla (id=-1) — Ecs. B.3–B.5 + Ec. B.4 (Kron)
        # ------------------------------------------------------------------
        elif ids[k] == -1:
            q_i = bus_idx[q]

            if p == 0:
                # Enlace entre barra q y el nodo de referencia — Ec. B.5
                Zll = xk + Z_work[q_i, q_i]
                dZ  = -Z_work[:, q_i].reshape(-1, 1)
            else:
                # Enlace entre barras p y q existentes — Ec. B.3
                p_i = bus_idx[p]
                Zll = (xk + Z_work[q_i, q_i] + Z_work[p_i, p_i]
                       - 2 * Z_work[p_i, q_i])
                dZ  = (Z_work[:, q_i] - Z_work[:, p_i]).reshape(-1, 1)

            # Reducción de Kron — Ec. B.4
            Z_work = Z_work - (dZ @ dZ.conj().T) / Zll

    # ------------------------------------------------------------------
    # Reordenar para que Z[i,j] corresponda a barras (i+1, j+1)
    # ------------------------------------------------------------------
    Z_out = np.zeros((nbus, nbus), dtype=complex)
    for bnum, idx in bus_idx.items():
        if 1 <= bnum <= nbus:
            for bnum2, idx2 in bus_idx.items():
                if 1 <= bnum2 <= nbus:
                    Z_out[bnum - 1, bnum2 - 1] = Z_work[idx, idx2]

    return Z_out


def imprimir_zbarra(Z: np.ndarray, nombres: dict, titulo: str = "Z_barra") -> None:
    """Imprime los elementos diagonales y algunas transferencias clave."""
    nbus = Z.shape[0]
    print(f"\n{'='*60}")
    print(f"  {titulo}  —  elementos diagonales (impedancias de Thévenin)")
    print(f"{'='*60}")
    print(f"  {'Barra':>5}  {'Nombre':<25}  {'|Zkk| [p.u.]':>13}  {'Xkk [p.u.]':>12}")
    print(f"  {'-'*5}  {'-'*25}  {'-'*13}  {'-'*12}")
    for k in range(nbus):
        Zkk = Z[k, k]
        print(f"  {k+1:>5}  {nombres.get(k+1,'?'):<25}  "
              f"{abs(Zkk):>13.5f}  {Zkk.imag:>12.5f}")


if __name__ == '__main__':
    import sys, os
    sys.path.insert(0, os.path.dirname(__file__))
    from sic_datos import DATA1, DATA0, NBUS, NOMBRES

    print("Construyendo Z_barra secuencia positiva …")
    Z1 = construir_zbarra(DATA1, NBUS)
    imprimir_zbarra(Z1, NOMBRES, "Z1_barra (secuencia positiva)")

    print("\nConstruyendo Z_barra secuencia cero …")
    Z0 = construir_zbarra(DATA0, NBUS)
    imprimir_zbarra(Z0, NOMBRES, "Z0_barra (secuencia cero)")


def aplicar_zkk_simulacion(Z: np.ndarray, Z1kk: list, Z0kk: list) -> tuple:
    """
    Reemplaza los elementos diagonales de Z1_barra y Z0_barra con los
    valores obtenidos de la simulación (Cuadro 5.1).

    Los elementos fuera de la diagonal (impedancias de transferencia Z_mk)
    se mantienen del algoritmo de red, que es la mejor estimación disponible.

    Parámetros
    ----------
    Z    : Z_barra (25×25) del algoritmo — solo se usa para off-diagonal
    Z1kk : lista de 25 valores Z1_kk desde Scc3 simulada
    Z0kk : lista de 25 valores Z0_kk desde Scc1 simulada / ratio red

    Retorna
    -------
    Z1, Z0 : matrices (25×25) con diagonal corregida
    """
    nbus = Z.shape[0]
    Z1 = Z.copy()
    Z0 = Z.copy()

    # Escalar off-diagonal proporcionalmente al cambio en diagonal
    # Z_mk_correg = Z_mk_algo × sqrt(Z1kk_sim[m] / Z1kk_algo[m])
    #                         × sqrt(Z1kk_sim[k] / Z1kk_algo[k])
    # (aproximación geométrica que preserva la simetría)
    Z1kk_algo = np.array([Z[k, k].imag for k in range(nbus)])
    Z0kk_algo = np.array([Z[k, k].imag for k in range(nbus)])  # misma base

    for m in range(nbus):
        f_m1 = np.sqrt(Z1kk[m] / Z1kk_algo[m]) if Z1kk_algo[m] > 0 else 1.0
        for k in range(nbus):
            if m == k:
                Z1[m, k] = 1j * Z1kk[m]
                Z0[m, k] = 1j * Z0kk[m]
            else:
                f_k1 = np.sqrt(Z1kk[k] / Z1kk_algo[k]) if Z1kk_algo[k] > 0 else 1.0
                Z1[m, k] = 1j * Z.imag[m, k] * f_m1 * f_k1
                Z0[m, k] = 1j * Z.imag[m, k] * f_m1 * f_k1  # misma escala approx

    return Z1, Z0


def calibrar_zbarra_sim(
    Z_algo: np.ndarray,
    Z1kk_sim: list,
    Z0kk_sim: list,
    V_sim_3f: dict,
    obs_buses: list = None,
    v_default: float = 0.9,
    Z_algo_0: np.ndarray = None,
) -> tuple:
    """
    Construye Z1_barra y Z0_barra calibradas con datos de simulación.

    Diagonal (exacta):
      Z1[k,k] = j × Z1kk_sim[k]    (de Scc3)
      Z0[k,k] = j × Z0kk_sim[k]    (de Scc1)

    Off-diagonal Z1_mk para filas de barras de observación — cada par NO
    ORDENADO {m,k} se calcula una sola vez (no depende del orden de
    iteración de obs_buses), con la siguiente precedencia:

      (a) Si (m,k) o (k,m) está en V_sim_3f → Z1_mk = (1-V_sim) × Z1kk_sim[lado con dato]
          EXACTO, desde simulación.
      (b) Si no hay dato Y k también es barra de observación → caso ambiguo:
          no existe un "lado k" único, porque ambas direcciones son válidas
          (V_mk con falla en k, V_km con falla en m). Se usa
          Z1_mk = (1-v_default) × min(Z1kk_sim[m], Z1kk_sim[k]), lo que
          garantiza V_mk ≥ v_default Y V_km ≥ v_default simultáneamente,
          sin depender de cuál de las dos barras se procesó primero.
      (c) Si no hay dato y k NO es barra de observación → caso simple,
          sin ambigüedad: Z1_mk = (1-v_default) × Z1kk_sim[k].

    Off-diagonal entre barras no-observación: del algoritmo (aceptable,
    solo afecta Z_pp en el denominador, no los valores de V en obs-buses).

    Off-diagonal Z0_mk: del algoritmo de secuencia cero (Z_algo_0, construido
    desde DATA0). Los pares con dato simulado se resuelven aguas abajo, en
    tensiones_falla.py, mediante Z_MK_0.

    ADVERTENCIA DE ESCALA (v1.0). La matriz algorítmica no incluye las
    fuentes de generación, por lo que su diagonal es entre 1 y 24 veces
    mayor que la obtenida de Scc. Al sustituir la diagonal por los valores
    simulados y conservar los elementos fuera de ella, la matriz resultante
    deja de cumplir |Z_mk| <= min(Z_mm, Z_kk). Por eso NO debe usarse un
    elemento algorítmico fuera de diagonal junto a uno calibrado en una
    misma expresión: es el origen del defecto corregido en v1.0 mediante
    el filtro de cobertura de tension_falla_linea_1f.

    Parámetros
    ----------
    Z_algo    : Z_barra del algoritmo, secuencia positiva (DATA1, 25×25)
    Z_algo_0  : Z_barra del algoritmo, secuencia cero (DATA0, 25×25).
                Si es None se reutiliza Z_algo (comportamiento de v1.0,
                incorrecto: la matriz homopolar quedaba siendo una copia
                de la de secuencia positiva).
    Z1kk_sim  : lista de 25 Z1_kk desde Scc3 simulada
    Z0kk_sim  : lista de 25 Z0_kk desde Scc1 simulada
    V_sim_3f  : dict {(m, k): V_mk} — tensiones 3φ simuladas
    obs_buses : barras de observación (default: [4, 15, 24])
    v_default : tensión límite para pares no calibrados (default 0.9,
                ver sic_datos.V_NO_CRITICO)

    Retorna
    -------
    Z1_cal, Z0_cal : matrices (25×25) calibradas
    """
    nbus = Z_algo.shape[0]
    Z1 = Z_algo.copy()
    Z0 = (Z_algo_0 if Z_algo_0 is not None else Z_algo).copy()
    obs_buses = obs_buses or [4, 15, 24]
    obs_set = set(obs_buses)

    # ── 1. Diagonal exacta desde simulación ──────────────────────────────
    for k in range(nbus):
        Z1[k, k] = 1j * Z1kk_sim[k]
        Z0[k, k] = 1j * Z0kk_sim[k]

    # ── 2. Filas de barras de observación: consistentes con Z_kk_sim ─────
    # Problema sin este paso: Z_mk del algoritmo es inconsistente con
    # Z_kk_sim → ratio Z_mk/Z_kk erróneo → tensiones de línea incorrectas.
    #
    # Cada par no-ordenado se calcula una sola vez (set `procesados`) para
    # eliminar la dependencia del orden de iteración — ver docstring (a)-(c).
    procesados = set()
    for m in obs_buses:
        m_i = m - 1
        for k_i in range(nbus):
            if m_i == k_i:
                continue
            k = k_i + 1
            par = frozenset((m, k))
            if par in procesados:
                continue  # ya calculado (y simétrico) en una iteración previa

            if (m, k) in V_sim_3f:
                z_mk = 1j * (1.0 - V_sim_3f[(m, k)]) * Z1kk_sim[k_i]
            elif (k, m) in V_sim_3f:
                z_mk = 1j * (1.0 - V_sim_3f[(k, m)]) * Z1kk_sim[m_i]
            elif k in obs_set:
                z_mk = 1j * (1.0 - v_default) * min(Z1kk_sim[m_i], Z1kk_sim[k_i])
            else:
                z_mk = 1j * (1.0 - v_default) * Z1kk_sim[k_i]

            Z1[m_i, k_i] = z_mk
            Z1[k_i, m_i] = z_mk              # simetría Z_mk = Z_km
            procesados.add(par)

    return Z1, Z0


def calibrar_zmk_1f_simulado(
    V_sim_1f: dict,
    Z1: np.ndarray,
    Z0: np.ndarray,
) -> dict:
    """
    Calibra una impedancia de transferencia efectiva Z1_mk, exclusiva
    para la rama de fallas monofásicas, directamente desde tensiones
    simuladas V_sim_1f (NUEVO en v1.0 — no existe en v1.0).

    Se obtiene invirtiendo la misma fórmula usada en
    tension_falla_barra_1f para reconstruir V^(1)_mk desde Z1_mk:

        V^(1)_mk = 1 - Z1_mk · I^(0)_fk ,   I^(0)_fk = 1 / (2·Z1_kk + Z0_kk)
        ⟹ Z1_mk_eff_1F = (1 - V_sim_1f) · (2·Z1_kk + Z0_kk)

    Z1_kk y Z0_kk son los elementos diagonales YA calibrados desde
    Scc3φ/Scc1φ (independientes de V_sim_1f).

    Este Z1_mk_eff_1F se usa luego en tension_falla_linea_1f en vez del
    Z1_mk calibrado solo con datos trifásicos (V_sim_3f) — sin este paso,
    el barrido de línea para fallas monofásicas no reproduce los valores
    simulados en los extremos de cada corredor (solo coincide la rama 3φ,
    porque Z1 sí queda calibrado con V_sim_3f en calibrar_zbarra_sim).

    Parámetros
    ----------
    V_sim_1f : dict {(m, k): V_mk} — tensiones 1φ simuladas (fase real)
    Z1, Z0   : matrices (25×25) ya calibradas (diagonal exacta)

    Retorna
    -------
    dict {(m, k): Z1_mk_eff_1F} — solo para los pares con dato simulado
    """
    Z1_eff = {}
    for (m, k), v in V_sim_1f.items():
        k_i = k - 1
        Z1kk = Z1[k_i, k_i]
        Z0kk = Z0[k_i, k_i]
        Z1_eff[(m, k)] = (1.0 - v) * (2 * Z1kk + Z0kk)
    return Z1_eff


## Módulo 3 — `cortocircuito.py`

In [ ]:
%%writefile cortocircuito.py
"""
cortocircuito.py — Potencias de Cortocircuito Trifásica y Monofásica
=====================================================================
Ecuaciones de la memoria (VR, 2026):

  Scc3φ = 1 / Z1_kk              (Ec. 3.33)  [p.u.] → × Sbase [MVA]
  Scc1φ = 1 / (2·Z1_kk + Z0_kk) (Ec. 3.34)  [p.u.] → × Sbase [MVA]

donde Z1_kk y Z0_kk son los elementos diagonales de las matrices de
impedancias de secuencia positiva y cero, respectivamente (= impedancias
equivalentes de Thévenin en la barra k).
"""

import numpy as np


def calcular_potencias_cc(
    Z1: np.ndarray,
    Z0: np.ndarray,
    Sbase: float = 100.0,
    nombres: dict = None,
) -> list[dict]:
    """
    Calcula Scc3φ y Scc1φ para cada barra.

    Parámetros
    ----------
    Z1, Z0 : ndarray (nbus, nbus), complex
        Matrices de impedancias de barra de secuencia positiva y cero.
    Sbase : float
        Potencia base en MVA (default 100 MVA).
    nombres : dict
        Mapa {numero_barra: nombre_barra}.

    Retorna
    -------
    Lista de dicts con claves:
      barra, nombre, Z1kk_pu, Z0kk_pu, X1kk_pu, X0kk_pu,
      Scc3_MVA, Scc1_MVA, Icc3_pu, Icc1_pu
    """
    nbus = Z1.shape[0]
    resultados = []

    for k in range(nbus):
        Z1kk = Z1[k, k]
        Z0kk = Z0[k, k]

        # Potencias de cortocircuito (Ecs. 3.33 y 3.34)
        Scc3 = Sbase / abs(Z1kk)          # MVA
        Scc1 = Sbase / abs(2*Z1kk + Z0kk) # MVA

        # Corrientes de cortocircuito en p.u. (con V_pref = 1 p.u.)
        Icc3 = 1.0 / abs(Z1kk)
        Icc1 = 3.0 / abs(2*Z1kk + Z0kk)  # corriente de secuencia cero × 3

        resultados.append({
            'barra'   : k + 1,
            'nombre'  : nombres.get(k + 1, f'Bus {k+1}') if nombres else f'Bus {k+1}',
            'Z1kk_pu' : Z1kk,
            'Z0kk_pu' : Z0kk,
            'X1kk_pu' : Z1kk.imag,
            'X0kk_pu' : Z0kk.imag,
            'Scc3_MVA': Scc3,
            'Scc1_MVA': Scc1,
            'Icc3_pu' : Icc3,
            'Icc1_pu' : Icc1,
        })

    return resultados


def imprimir_tabla_cc(resultados: list[dict], Sbase: float = 100.0) -> None:
    """Imprime tabla de resultados de cortocircuito."""
    print(f"\n{'='*80}")
    print("  POTENCIAS DE CORTOCIRCUITO — STN 25 barras")
    print(f"  Base: Sbase = {Sbase:.0f} MVA")
    print(f"{'='*80}")
    print(f"  {'#':>3}  {'Barra':<24}  {'X1kk':>8}  {'X0kk':>8}  "
          f"{'Scc3 [MVA]':>11}  {'Scc1 [MVA]':>11}")
    print(f"  {'-'*3}  {'-'*24}  {'-'*8}  {'-'*8}  {'-'*11}  {'-'*11}")

    for r in resultados:
        print(f"  {r['barra']:>3}  {r['nombre']:<24}  "
              f"{r['X1kk_pu']:>8.5f}  {r['X0kk_pu']:>8.5f}  "
              f"{r['Scc3_MVA']:>11.1f}  {r['Scc1_MVA']:>11.1f}")


if __name__ == '__main__':
    import sys, os
    sys.path.insert(0, os.path.dirname(__file__))
    from sic_datos import DATA1, DATA0, NBUS, NOMBRES, SBASE
    from zbarra   import construir_zbarra

    Z1 = construir_zbarra(DATA1, NBUS)
    Z0 = construir_zbarra(DATA0, NBUS)
    res = calcular_potencias_cc(Z1, Z0, Sbase=SBASE, nombres=NOMBRES)
    imprimir_tabla_cc(res, Sbase=SBASE)


## Módulo 4 — `tensiones_falla.py`

In [ ]:
%%writefile tensiones_falla.py
"""
tensiones_falla.py — Tensiones de Falla en Barras y Tramos de Línea
====================================================================
Calcula la tensión en barras de observación (Cardones, Alto Jahuel,
Valdivia) ante fallas de cortocircuito en cualquier punto del sistema.

Ecuaciones de la memoria (VR, 2026):

FALLA TRIFÁSICA EN BARRA k — Ec. 3.8:
  V_mk = 1 − Z1_mk / Z1_kk

FALLA MONOFÁSICA EN BARRA k — Ec. 3.13 (fórmula exacta de tres secuencias):
  V_a,mk = 1 − (2·Z1_mk + Z0_mk) / (2·Z1_kk + Z0_kk)
  ← tensión de la FASE FALLADA, no de secuencia positiva. Es la misma
    magnitud con que se determinó el criterio de susceptibilidad del
    inversor por simulación dinámica (§3.4.2 de la memoria).

FALLA TRIFÁSICA EN PUNTO p DE LÍNEA k–j — Ecs. 3.16, 3.8:
  Z1_mp = (1−ξ)·Z1_mk + ξ·Z1_mj
  Z1_pp = (1−ξ)²·Z1_kk + ξ²·Z1_jj + 2ξ(1−ξ)·Z1_kj + ξ(1−ξ)·z1_kj
  V_mp  = 1 − Z1_mp / Z1_pp
  con ξ = Lkp / Lkj ∈ [0,1]

FALLA MONOFÁSICA EN PUNTO p DE LÍNEA k–j — Ecs. 3.16, 3.13:
  (mismas fórmulas geométricas para Z0_mp y Z0_pp)
  V_a,mp = 1 − (2·Z1_mp + Z0_mp) / (2·Z1_pp + Z0_pp)

CAMBIOS v1.0
-------------
  - Filtro de cobertura en tension_falla_linea_1f: el cálculo monofásico
    sobre un corredor exige Z0_mk simulado en AMBOS extremos. Interpolar
    entre un extremo calibrado y otro algorítmico mezcla dos escalas
    distintas y producía perfiles imposibles (mínimos interiores cercanos
    a 0 p.u. en corredores cuyos dos extremos superaban 0,94 p.u.).
  - El clamp a [0,1] deja de ser silencioso: emite aviso por consola.
  - Rótulos y docstrings alineados con el criterio real del código, que es
    de desigualdad ESTRICTA (V < v_umbral).
"""

import numpy as np
from sic_datos import V_UMBRAL, V_NO_CRITICO


# ===========================================================================
# FALLAS EN BARRAS
# ===========================================================================

def tension_falla_barra_3f(Z1: np.ndarray, obs_buses: dict,
                           v_sim: dict = None) -> dict:
    """
    Tensión en barras de observación m ante falla trifásica en barra k.

    Si v_sim contiene el par (m, k) → usa el valor simulado directamente.
    En caso contrario → calcula V_mk = 1 - Z_mk/Z_kk (del algoritmo).

    Retorna dict {m: {k: |V_mk|}}
    """
    nbus = Z1.shape[0]
    v_sim = v_sim or {}
    res = {}
    for m in obs_buses:
        m_i = m - 1
        res[m] = {}
        for k in range(1, nbus + 1):
            if (m, k) in v_sim:
                res[m][k] = v_sim[(m, k)]          # exacto de simulación
            else:
                k_i = k - 1
                Zmk = Z1[m_i, k_i]
                Zkk = Z1[k_i, k_i]
                res[m][k] = abs(1.0 - Zmk / Zkk)   # calculado de Z_barra
    return res


def tension_falla_barra_1f(Z1: np.ndarray, Z0: np.ndarray,
                            obs_buses: dict,
                            z_mk_1: dict = None,
                            z_mk_0: dict = None,
                            v_sim_1f: dict = None,
                            v_sim_3f: dict = None) -> dict:
    """
    Tensión de fase real en barras de observación m ante falla 1φ en k.

    Fórmula exacta de tres secuencias — NUEVO:
      V_a_mk = 1 - (2·Z1_mk + Z0_mk) / (2·Z1_kk + Z0_kk)

    Jerarquía de fuentes para Z1_mk y Z0_mk:
      1. z_mk_1[(m,k)] y z_mk_0[(m,k)] — datos simulados directos
         (ZMK_1 y Z_MK_0 de sic_datos). Caso preferido: Z1_mk igual al
         de la rama 3φ (no se inventa un segundo valor), Z0_mk real.
      2. Si falta z_mk_0 pero hay v_sim_3f[(m,k)]: recalibra Z1_mk desde
         ese dato y usa Z0[m,k] del algoritmo como respaldo.
      3. Fallback puro: Z1[m,k] y Z0[m,k] del algoritmo de red.

    Retorna dict {m: {k: |V_a_mk|}}
    """
    nbus     = Z1.shape[0]
    z_mk_1   = z_mk_1   or {}
    z_mk_0   = z_mk_0   or {}
    v_sim_1f = v_sim_1f or {}
    v_sim_3f = v_sim_3f or {}

    res = {}
    for m in obs_buses:
        m_i = m - 1
        res[m] = {}
        for k in range(1, nbus + 1):
            k_i  = k - 1
            Z1kk = Z1[k_i, k_i]
            Z0kk = Z0[k_i, k_i]

            if (m, k) in z_mk_1 and (m, k) in z_mk_0:
                # Caso 1: formula exacta con datos simulados
                Z1mk = 1j * z_mk_1[(m, k)]
                Z0mk = 1j * z_mk_0[(m, k)]
                res[m][k] = abs(1.0 - (2*Z1mk + Z0mk) / (2*Z1kk + Z0kk))
            elif (m, k) in v_sim_1f:
                # Caso 2a: dato de tension directa (respaldo)
                res[m][k] = v_sim_1f[(m, k)]
            else:
                # Caso 2b/3: Z1_mk desde V_sim_3f o algoritmo; Z0_mk algoritmo
                if (m, k) in v_sim_3f:
                    Z1mk = 1j * (1.0 - v_sim_3f[(m, k)]) * Z1kk.imag
                else:
                    Z1mk = Z1[m_i, k_i]
                Z0mk = Z0[m_i, k_i]
                Ifk1 = 1.0 / (2*Z1kk + Z0kk)
                res[m][k] = abs(1.0 - Z1mk * Ifk1)
    return res


# ===========================================================================
# FALLAS EN TRAMOS DE LÍNEA
# ===========================================================================

def tension_falla_linea_3f(
    Z1: np.ndarray,
    corredores: list,
    obs_buses: dict,
    xi_vals: np.ndarray = None,
    v_sim: dict = None,
    zkj_sim: dict = None,
    v_umbral: float = V_UMBRAL,
) -> dict:
    """
    Tensión en barras de observación m ante falla 3φ en punto p de línea k–j.

    Numerador Z_mp = (1-ξ)Z_mk + ξZ_mj  → exacto desde V_sim (calibrado)
    Denominador Z_pp usa Z_kj exacto desde zkj_sim si disponible,
    algoritmo en caso contrario.

    Lógica de cortocircuito:
      - Si ambos extremos V_mk_eff > v_umbral → tramo no crítico (V=V_NO_CRITICO)
      - Si al menos un extremo es crítico → calcula con Ec. 3.16
    """
    if xi_vals is None:
        xi_vals = np.linspace(0.0, 1.0, 200)  # curva continua
    v_sim  = v_sim  or {}
    zkj_sim = zkj_sim or {}

    res = {}
    for m in obs_buses:
        m_i = m - 1
        res[m] = {}
        for c in corredores:
            k, j   = c['from'], c['to']
            k_i, j_i = k - 1, j - 1
            label  = c['label']

            # Z_kj: exacto desde simulación si disponible, algoritmo si no
            # Z_kj: impedancia de transferencia desde simulación (Ec. 3.41)
            if (k, j) in zkj_sim:
                Z1kj_sim = 1j * zkj_sim[(k, j)]
            else:
                Z1kj_sim = Z1[k_i, j_i]   # fallback: elemento off-diagonal Z_barra

            # z_kj: impedancia SERIE de un circuito individual (Ec. 3.16)
            z1_kj_serie = 1j * c['x1_pu_circ']

            Z1kk = Z1[k_i, k_i]
            Z1jj = Z1[j_i, j_i]

            # Tensiones efectivas en extremos
            v_mk = v_sim.get((m, k), V_NO_CRITICO)
            v_mj = v_sim.get((m, j), V_NO_CRITICO)

            if v_mk >= v_umbral and v_mj >= v_umbral:
                res[m][label] = [(xi, float("inf")) for xi in xi_vals]
                continue

            pts = []
            for xi in xi_vals:
                Z1mk_xi = Z1[m_i, k_i]
                Z1mj_xi = Z1[m_i, j_i]
                Z1mp = (1 - xi)*Z1mk_xi + xi*Z1mj_xi
                Z1pp = ((1-xi)**2*Z1kk + xi**2*Z1jj
                        + 2*xi*(1-xi)*Z1kj_sim       # transferencia desde simulación
                        + xi*(1-xi)*z1_kj_serie)     # serie circuito individual
                Vmp  = 1.0 - Z1mp / Z1pp
                pts.append((xi, abs(Vmp)))
            res[m][label] = pts
    return res


def tension_falla_linea_1f(
    Z1: np.ndarray,
    Z0: np.ndarray,
    corredores: list,
    obs_buses: dict,
    xi_vals: np.ndarray = None,
    v_sim_3f: dict = None,
    v_sim_1f: dict = None,
    zkj_sim: dict = None,
    v_umbral: float = V_UMBRAL,
    z_mk_1: dict = None,
    z_mk_0: dict = None,
) -> dict:
    """
    Tensión de fase real en barra de observación m ante falla 1φ en línea k–j.

    Fórmula exacta de tres secuencias:
      Z1_mp = (1-ξ)·Z1_mk + ξ·Z1_mj          (numerador sec. positiva)
      Z0_mp = (1-ξ)·Z0_mk + ξ·Z0_mj          (numerador sec. cero)
      Z1_pp, Z0_pp = interpolación cuadrática habitual
      I_fp1 = 1 / (2·Z1_pp + Z0_pp)
      V_a_mp = 1 - (2·Z1_mp + Z0_mp) · I_fp1

    z_mk_1 y z_mk_0 (NUEVO v1.0): impedancias de transferencia simuladas
      (Z_MK_1, Z_MK_0 de sic_datos). Cuando están disponibles para los
      extremos k y j del corredor, reemplazan los elementos off-diagonal
      de Z_barra (que son del algoritmo), igual que calibrar_zbarra_sim
      ya hace con Z1_mk para la rama 3φ. Eliminan el parche Z1_eff_1f
      de v1.0, que era metodológicamente inconsistente (creaba un segundo
      valor de Z1_mk distinto al de la rama 3φ).

    Si faltan z_mk_0 para algún extremo, usa Z0[m,k] del algoritmo.
    Si faltan z_mk_1 para algún extremo, usa Z1[m,k] del algoritmo.
    """
    if xi_vals is None:
        xi_vals = np.linspace(0.0, 1.0, 200)
    v_sim_3f = v_sim_3f or {}
    v_sim_1f = v_sim_1f or {}
    zkj_sim  = zkj_sim  or {}
    z_mk_1   = z_mk_1   or {}
    z_mk_0   = z_mk_0   or {}

    def _v_eff_1f(m, k):
        """Tensión efectiva en extremo de corredor — para filtro de criticidad."""
        if (m, k) in v_sim_1f: return v_sim_1f[(m, k)]
        if (m, k) in v_sim_3f: return v_sim_3f[(m, k)]
        return V_NO_CRITICO

    res = {}
    for m in obs_buses:
        m_i = m - 1
        res[m] = {}
        for c in corredores:
            k, j   = c['from'], c['to']
            k_i, j_i = k - 1, j - 1
            label  = c['label']

            # Z_kj transferencia desde simulación (Ec. 3.41)
            if (k, j) in zkj_sim:
                Z1kj_sim = 1j * zkj_sim[(k, j)]
                Z0kj_sim = 1j * zkj_sim[(k, j)]
            else:
                Z1kj_sim = Z1[k_i, j_i]
                Z0kj_sim = Z0[k_i, j_i]

            z1_kj_serie = 1j * c['x1_pu_circ']
            z0_kj_serie = 1j * c['x0_pu_circ']

            Z1kk = Z1[k_i, k_i];  Z1jj = Z1[j_i, j_i]
            Z0kk = Z0[k_i, k_i];  Z0jj = Z0[j_i, j_i]

            # ── Filtro de cobertura (v1.0) ──────────────────────────
            # La interpolación de Z0_mp exige dato simulado en AMBOS
            # extremos. Si falta en uno, la interpolación mezclaría un
            # valor calibrado con uno algorítmico de escala distinta.
            if (m, k) not in z_mk_0 or (m, j) not in z_mk_0:
                res[m][label] = [(xi, float("inf")) for xi in xi_vals]
                continue

            # Filtro de criticidad: misma lógica que en v1.0
            v_mk = _v_eff_1f(m, k)
            v_mj = _v_eff_1f(m, j)
            if v_mk >= v_umbral and v_mj >= v_umbral:
                res[m][label] = [(xi, float("inf")) for xi in xi_vals]
                continue

            # Z1_mk/Z1_mj: dato simulado si disponible, algoritmo si no
            Z1mk = 1j * z_mk_1[(m, k)] if (m, k) in z_mk_1 else Z1[m_i, k_i]
            Z1mj = 1j * z_mk_1[(m, j)] if (m, j) in z_mk_1 else Z1[m_i, j_i]

            # Z0_mk/Z0_mj: dato simulado si disponible, algoritmo si no
            Z0mk = 1j * z_mk_0[(m, k)] if (m, k) in z_mk_0 else Z0[m_i, k_i]
            Z0mj = 1j * z_mk_0[(m, j)] if (m, j) in z_mk_0 else Z0[m_i, j_i]

            pts = []
            v_max_bruto = 0.0
            for xi in xi_vals:
                Z1mp = (1-xi)*Z1mk + xi*Z1mj
                Z1pp = ((1-xi)**2*Z1kk + xi**2*Z1jj
                        + 2*xi*(1-xi)*Z1kj_sim
                        + xi*(1-xi)*z1_kj_serie)
                Z0mp = (1-xi)*Z0mk + xi*Z0mj
                Z0pp = ((1-xi)**2*Z0kk + xi**2*Z0jj
                        + 2*xi*(1-xi)*Z0kj_sim
                        + xi*(1-xi)*z0_kj_serie)
                denom = 2*Z1pp + Z0pp
                if abs(denom) < 1e-8:          # guarda numerica: polo espurio
                    pts.append((xi, V_NO_CRITICO))
                    continue
                Ifp1  = 1.0 / denom
                V_amp = 1.0 - (2*Z1mp + Z0mp) * Ifp1
                v_amp = abs(V_amp)
                v_max_bruto = max(v_max_bruto, v_amp)
                pts.append((xi, min(v_amp, 1.0)))      # clamp fisico [0,1]
            if v_max_bruto > 1.0 + 1e-6:               # no enmascarar (v1.0)
                print(f"  [aviso] V_a > 1 p.u. en {label} (m={m}): "
                      f"maximo {v_max_bruto:.4f} p.u.")
            res[m][label] = pts
    return res


# ===========================================================================
# IDENTIFICACIÓN DE ÁREAS DE VULNERABILIDAD
# ===========================================================================

def areas_vulnerabilidad_barras(
    v3f: dict, v1f: dict, obs_buses: dict,
    nombres: dict, v_umbral: float = V_UMBRAL,
) -> dict:
    """
    Identifica barras cuya falla produce V_mk < v_umbral (desigualdad
    estricta) en la barra de observación m.

    Retorna
    -------
    dict {m: {'3f': [lista_barras_criticas], '1f': [lista_barras_criticas]}}
    """
    areas = {}
    for m, m_nombre in obs_buses.items():
        criticas_3f = [k for k, v in v3f[m].items() if v < v_umbral]
        criticas_1f = [k for k, v in v1f[m].items() if v < v_umbral]
        areas[m] = {'3f': criticas_3f, '1f': criticas_1f}
    return areas


def areas_vulnerabilidad_lineas(
    vl3f: dict, vl1f: dict, obs_buses: dict, v_umbral: float = V_UMBRAL,
) -> dict:
    """
    Identifica corredores cuya falla (en algún ξ) produce V_mp < v_umbral
    (desigualdad estricta).

    Retorna
    -------
    dict {m: {'3f': [label_criticos], '1f': [label_criticos]}}
    """
    areas = {}
    for m in obs_buses:
        criticas_3f = [lbl for lbl, pts in vl3f[m].items()
                       if any(v < v_umbral for _, v in pts)]
        criticas_1f = [lbl for lbl, pts in vl1f[m].items()
                       if any(v < v_umbral for _, v in pts)]
        areas[m] = {'3f': criticas_3f, '1f': criticas_1f}
    return areas


def curvas_area_vulnerabilidad(
    vl3f: dict, vl1f: dict,
    areas_l: dict, corredores: list,
    obs_buses: dict,
) -> dict:
    """
    Para cada barra de observación m, retorna las curvas V(ξ) completas
    solo para los corredores críticos de su área de vulnerabilidad.

    Los puntos se entregan como (ξ, V), con ξ ∈ [0,1]; longitud_km se
    adjunta por separado en la clave 'L_km'.

    Retorna
    -------
    dict {m: [{'label', 'L_km', 'lambda_anual',
               'critico_3f', 'critico_1f',
               'pts_3f': [(km, V), ...],
               'pts_1f': [(km, V), ...]}]}
    """
    corr_dict = {c['label']: c for c in corredores}
    result = {}
    for m in obs_buses:
        criticos_3f = set(areas_l[m]['3f'])
        criticos_1f = set(areas_l[m]['1f'])
        criticos = criticos_3f | criticos_1f
        curvas = []
        for lbl in criticos:
            c = corr_dict[lbl]
            L = c['longitud_km']
            pts3 = list(vl3f[m][lbl])   # ya son (xi, V)
            pts1 = list(vl1f[m][lbl])
            curvas.append({
                'label':       lbl,
                'L_km':        L,
                'lambda_anual': 0.7 * L / 100.0,
                'critico_3f':  lbl in criticos_3f,
                'critico_1f':  lbl in criticos_1f,
                'pts_3f':      pts3,
                'pts_1f':      pts1,
            })
        result[m] = curvas
    return result




def imprimir_tensiones_barras(
    v3f: dict, v1f: dict,
    obs_buses: dict, nombres: dict, v_umbral: float = V_UMBRAL,
) -> None:
    """Tabla de tensiones de falla por barra."""
    nbus = len(v3f[list(obs_buses.keys())[0]])
    for m, m_nombre in obs_buses.items():
        print(f"\n{'='*70}")
        print(f"  Barra de observación: {m_nombre}  (barra {m})")
        print(f"  Umbral de conmutación: V < {v_umbral} p.u.")
        print(f"{'='*70}")
        print(f"  {'k':>3}  {'Barra de falla':<24}  "
              f"{'|V_mk| 3φ':>10}  {'|V_a,mk| 1φ':>13}  "
              f"{'3φ':>4}  {'1φ':>4}")
        print(f"  {'-'*3}  {'-'*24}  {'-'*10}  {'-'*13}  {'-'*4}  {'-'*4}")
        for k in range(1, nbus + 1):
            v3 = v3f[m][k]
            v1 = v1f[m][k]
            flag3 = '⚠' if v3 < v_umbral else ''
            flag1 = '⚠' if v1 < v_umbral else ''
            s3 = f'>{v_umbral}' if not np.isfinite(v3) or v3 >= 0.8999 else f'{v3:>10.4f}'
            s1 = f'>{v_umbral}' if not np.isfinite(v1) or v1 >= 0.8999 else f'{v1:>10.4f}'
            print(f"  {k:>3}  {nombres.get(k,'?'):<24}  "
                  f"{s3:>10}  {s1:>13}  {flag3:>4}  {flag1:>4}")


def imprimir_tensiones_lineas(
    vl3f: dict, vl1f: dict,
    obs_buses: dict, v_umbral: float = V_UMBRAL,
) -> None:
    """Resumen de tensiones mínimas por corredor de línea."""
    for m, m_nombre in obs_buses.items():
        print(f"\n{'='*80}")
        print(f"  Barra de observación: {m_nombre}  — Fallas en tramos de línea")
        print(f"{'='*80}")
        print(f"  {'Corredor':<40}  "
              f"{'Vmin 3φ':>8}  {'ξ':>5}  "
              f"{'Vmin 1φ':>8}  {'ξ':>5}  {'Crítico':>7}")
        print(f"  {'-'*40}  {'-'*8}  {'-'*5}  {'-'*8}  {'-'*5}  {'-'*7}")
        for lbl in vl3f[m]:
            pts3 = vl3f[m][lbl]
            pts1 = vl1f[m][lbl]
            vmin3, xi3 = min(pts3, key=lambda p: p[1])
            vmin1, xi1 = min(pts1, key=lambda p: p[1])
            xi3_val   = min(pts3, key=lambda p: p[1])[0]
            vmin3_val = min(pts3, key=lambda p: p[1])[1]
            xi1_val   = min(pts1, key=lambda p: p[1])[0]
            vmin1_val = min(pts1, key=lambda p: p[1])[1]
            critico = '⚠' if vmin3_val < v_umbral or vmin1_val < v_umbral else ''
            # Formatear: inf → ">umbral"
            s3 = f'>{v_umbral}' if not np.isfinite(vmin3_val) else f'{vmin3_val:>8.4f}'
            s1 = f'>{v_umbral}' if not np.isfinite(vmin1_val) else f'{vmin1_val:>8.4f}'
            x3 = f'{xi3_val:>5.2f}' if np.isfinite(vmin3_val) else '  ---'
            x1 = f'{xi1_val:>5.2f}' if np.isfinite(vmin1_val) else '  ---'
            print(f"  {lbl:<40}  {s3:>8}  {x3}  {s1:>8}  {x1}  {critico:>7}")


def imprimir_areas_vulnerabilidad(
    areas_b: dict, areas_l: dict,
    obs_buses: dict, nombres: dict,
) -> None:
    """
    Resumen de áreas de vulnerabilidad — formato tabular unificado.

    En vez de listar las barras/líneas críticas dos veces (una para 3φ,
    otra para 1φ, con la lista 1φ casi siempre subconjunto de la 3φ), se
    presenta una sola tabla por barra de observación, con columnas que
    indican si cada elemento es crítico bajo cada tipo de falla.
    """
    ANCHO = 72

    for m, m_nombre in obs_buses.items():
        crit_b3 = set(areas_b[m]['3f'])
        crit_b1 = set(areas_b[m]['1f'])
        crit_l3 = list(areas_l[m]['3f'])
        crit_l1 = set(areas_l[m]['1f'])

        # Unión preservando orden de aparición (3φ primero, luego 1φ-only)
        todas_b = sorted(crit_b3 | crit_b1)
        todas_l = list(crit_l3) + [lbl for lbl in crit_l1 if lbl not in crit_l3]

        print(f"\n{'='*ANCHO}")
        print(f"  {m_nombre}  (barra {m})")
        print('='*ANCHO)
        print(f"  Barras críticas:  3φ = {len(crit_b3):2d}   1φ = {len(crit_b1):2d}")
        print(f"  Líneas críticas:  3φ = {len(crit_l3):2d}   1φ = {len(crit_l1):2d}")

        print(f"\n  {'Barra':<26}{'3φ':>6}{'1φ':>6}")
        print(f"  {'-'*26}{'-'*6}{'-'*6}")
        if todas_b:
            for k in todas_b:
                c3 = '   X' if k in crit_b3 else '    '
                c1 = '   X' if k in crit_b1 else '    '
                print(f"  {k:>3d}  {nombres.get(k,'?'):<21}{c3:>6}{c1:>6}")
        else:
            print("  (ninguna)")

        print(f"\n  {'Corredor':<42}{'3φ':>6}{'1φ':>6}")
        print(f"  {'-'*42}{'-'*6}{'-'*6}")
        if todas_l:
            for lbl in todas_l:
                c3 = '   X' if lbl in crit_l3 else '    '
                c1 = '   X' if lbl in crit_l1 else '    '
                print(f"  {lbl:<42}{c3:>6}{c1:>6}")
        else:
            print("  (ninguna)")
    print(f"\n{'='*ANCHO}")


if __name__ == '__main__':
    import sys, os
    sys.path.insert(0, os.path.dirname(__file__))
    from sic_datos   import DATA1, DATA0, NBUS, NOMBRES, BARRAS_OBS, V_UMBRAL, get_corredores
    from zbarra      import construir_zbarra

    Z1 = construir_zbarra(DATA1, NBUS)
    Z0 = construir_zbarra(DATA0, NBUS)
    corredores = get_corredores()
    xi = np.linspace(0.0, 1.0, 21)

    v3f  = tension_falla_barra_3f(Z1, BARRAS_OBS)
    v1f  = tension_falla_barra_1f(Z1, Z0, BARRAS_OBS)
    vl3f = tension_falla_linea_3f(Z1, corredores, BARRAS_OBS, xi)
    vl1f = tension_falla_linea_1f(Z1, Z0, corredores, BARRAS_OBS, xi)

    imprimir_tensiones_barras(v3f, v1f, BARRAS_OBS, NOMBRES, V_UMBRAL)
    imprimir_tensiones_lineas(vl3f, vl1f, BARRAS_OBS, V_UMBRAL)

    areas_b = areas_vulnerabilidad_barras(v3f, v1f, BARRAS_OBS, NOMBRES, V_UMBRAL)
    areas_l = areas_vulnerabilidad_lineas(vl3f, vl1f, BARRAS_OBS, V_UMBRAL)
    imprimir_areas_vulnerabilidad(areas_b, areas_l, BARRAS_OBS, NOMBRES)


## Módulo 5 — `graficos.py`

In [ ]:
%%writefile graficos.py
"""
graficos.py — Gráficos de áreas de vulnerabilidad y frecuencia de fallas
=========================================================================
Sigue el esquema de la memoria (Figuras 4.5 y 4.6):

  Fig. 4.5 izq: V vs ξ (perfil de tensión continuo por tramo)
  Fig. 4.5 der: fdp f(V) = |dξ/dV| (densidad de probabilidad de tensión)
  Fig. 4.6:     histograma fallas/año por bin de tensión (3φ y 1φ)

Tasas de falla:
  Líneas: LAMBDA_LINEA = 0.7 fallas/año por 100 km  (total)
  Barras: LAMBDA_BARRA = 0.08 fallas/año por barra   (total)
  F_3F = 0.05  fracción trifásicas
  F_1F = 0.80  fracción monofásicas
"""

import os
import numpy as np
import matplotlib
matplotlib.rcParams.update({
    # Fuente tipo Times — el mismo aspecto que produce \usepackage{mathptmx}
    # en la memoria (TeX Gyre Termes / Liberation Serif son clones
    # métricamente compatibles con Times New Roman; se usa el primero
    # disponible en el sistema)
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'Liberation Serif',
                           'Nimbus Roman', 'TeX Gyre Termes', 'DejaVu Serif'],
    'mathtext.fontset':  'stix',   # más cercano a Times que 'cm' (Computer Modern)
    # Tamaños
    'font.size':         10,
    'axes.titlesize':    11,
    'axes.labelsize':    10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
    'legend.fontsize':   8,
    # Líneas y spines
    'axes.linewidth':    0.7,
    'lines.linewidth':   1.5,
    'patch.linewidth':   0.5,
    # Recuadro completo en los 4 lados, como el axis de pgfplots
    'axes.spines.top':   True,
    'axes.spines.right': True,
    # Grid sólido (equivalente a grid=major de pgfplots), no punteado
    'axes.grid':         True,
    'grid.linewidth':    0.4,
    'grid.alpha':        0.55,
    'grid.linestyle':    '-',
    'grid.color':        '0.75',
    'axes.edgecolor':    'black',
    # Figura
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'savefig.pad_inches': 0.05,
    # PDF vectorial con fuentes embebidas
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})
import matplotlib.pyplot as plt

from sic_datos import V_UMBRAL_3F, V_TRANS_1F_HI, V_TRANS_1F_LO

# Paleta alineada con la Figura de validación (tikz): colores primarios
# saturados en vez de la paleta pastel anterior, para que los gráficos del
# notebook luzcan consistentes con las figuras hechas a mano en LaTeX.
COLOR_1F_LINEA = 'tab:blue'
COLOR_3F_LINEA = 'tab:red'
COLOR_1F_BARRA = 'tab:green'
COLOR_3F_BARRA = 'tab:orange'
COLOR_UMBRAL   = 'purple'    # mismo color que la línea de "límite de tensión"

LAMBDA_LINEA  = 0.7
LAMBDA_BARRA  = 0.08
F_3F          = 0.05
F_1F          = 0.80
_FIG_DIR      = 'figuras_sic'

# Modelos de probabilidad de FC: V_UMBRAL_3F, V_TRANS_1F_HI, V_TRANS_1F_LO
# se importan desde sic_datos.py (fuente única de verdad, ec:pfc_trifasica / ec:pfc_monofasica)


def _pfc(v, tipo):
    """Probabilidad de FC dado tensión v y tipo de falla ('3f' o '1f')."""
    if tipo == '3f':
        return 1.0 if v < V_UMBRAL_3F else 0.0
    else:  # 1f
        if v >= V_TRANS_1F_HI:  return 0.0
        if v <= V_TRANS_1F_LO:  return 1.0
        return (V_TRANS_1F_HI - v) / (V_TRANS_1F_HI - V_TRANS_1F_LO)
N_BINS       = 20    # bins de 0.05 → 0.80, 0.85 y 0.90 caen exactamente en bordes


def _mostrar(fig, nombre_archivo):
    """Guarda la figura como PDF vectorial y la muestra en notebook si es posible."""
    nombre_pdf = nombre_archivo.replace('.png', '.pdf')
    os.makedirs(_FIG_DIR, exist_ok=True)
    ruta = os.path.join(_FIG_DIR, nombre_pdf)
    fig.savefig(ruta, format='pdf', bbox_inches='tight', pad_inches=0.05)
    print(f"  Guardada: {ruta}")
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        pass
    plt.close(fig)


def _calcular_fdp(pts):
    """
    Calcula la fdp f(V) = |dξ/dV| numéricamente a partir de la curva V(ξ).

    Usa diferencias centrales para estimar dV/dξ en cada punto interior,
    y diferencias unilaterales en los extremos.

    Retorna (V_arr, fdp_arr) — arrays paralelos, ordenados por V creciente.
    """
    xi_arr = np.array([p[0] for p in pts])
    V_arr  = np.array([p[1] for p in pts])

    # dV/dξ por diferencias finitas
    dVdxi = np.gradient(V_arr, xi_arr)

    # |dξ/dV| = 1/|dV/dξ|, evitar división por cero
    with np.errstate(divide='ignore', invalid='ignore'):
        fdp = np.where(np.abs(dVdxi) > 1e-9, 1.0 / np.abs(dVdxi), np.nan)

    # Ordenar por V para graficar correctamente
    orden  = np.argsort(V_arr)
    return V_arr[orden], fdp[orden]


def _pfc_integral(v_lo, v_hi, tipo):
    """
    Integral de P_FC(v) sobre el intervalo [v_lo, v_hi], dividida por (v_hi - v_lo).
    Equivale al valor medio de P_FC en el bin — exacto para funciones lineales por partes.
    """
    if v_hi <= v_lo:
        return 0.0
    dv = v_hi - v_lo

    if tipo == '3f':
        # Escalón en V_UMBRAL_3F
        if v_hi <= V_UMBRAL_3F:   return 1.0   # todo el bin con P_FC=1
        if v_lo >= V_UMBRAL_3F:   return 0.0   # todo el bin con P_FC=0
        # bin cruza el umbral
        return (V_UMBRAL_3F - v_lo) / dv

    else:  # 1f — lineal entre V_TRANS_1F_LO y V_TRANS_1F_HI
        hi, lo = V_TRANS_1F_HI, V_TRANS_1F_LO
        span = hi - lo

        def _pfc_val(v):
            if v >= hi:  return 0.0
            if v <= lo:  return 1.0
            return (hi - v) / span

        # Integral de P_FC sobre [v_lo, v_hi] usando trapecios exactos en los nodos
        # donde P_FC cambia de pendiente (v=lo y v=hi)
        nodes = sorted(set([v_lo, v_hi] + [x for x in [lo, hi] if v_lo < x < v_hi]))
        integral = 0.0
        for a, b in zip(nodes[:-1], nodes[1:]):
            integral += 0.5 * (_pfc_val(a) + _pfc_val(b)) * (b - a)
        return integral / dv


def _histograma_fallas_raw(pts, lambda_total, n_bins=N_BINS):
    V_arr  = np.array([p[1] for p in pts])
    V_arr  = V_arr[np.isfinite(V_arr)]
    bordes = np.linspace(0.0, 1.0, n_bins + 1)
    counts = np.zeros(n_bins)
    n      = len(V_arr)
    if n == 0:
        return bordes, counts
    for i in range(n_bins):
        frac = np.sum((V_arr >= bordes[i]) & (V_arr < bordes[i+1])) / n
        counts[i] = frac * lambda_total
    return bordes, counts


def _histograma_barras_raw(barras_crit, v_dict, lambda_por_barra, n_bins=N_BINS):
    """Histograma de hundimientos de tensión/año en barras (sin P_FC)."""
    bordes = np.linspace(0.0, 1.0, n_bins + 1)
    counts = np.zeros(n_bins)
    ancho  = bordes[1] - bordes[0]
    for k in barras_crit:
        v = v_dict[k]
        if not np.isfinite(v):
            continue
        bi = min(int(v / ancho), n_bins - 1)
        counts[bi] += lambda_por_barra
    return bordes, counts


def _histograma_fallas(pts, lambda_total, tipo, n_bins=N_BINS):
    """Histograma de FC/año por bin, ponderado por integral exacta de P_FC."""
    V_arr  = np.array([p[1] for p in pts])
    V_arr  = V_arr[np.isfinite(V_arr)]
    bordes = np.linspace(0.0, 1.0, n_bins + 1)
    counts = np.zeros(n_bins)
    n      = len(V_arr)
    if n == 0:
        return bordes, counts
    for i in range(n_bins):
        frac   = np.sum((V_arr >= bordes[i]) & (V_arr < bordes[i+1])) / n
        p_mean = _pfc_integral(bordes[i], bordes[i+1], tipo)
        counts[i] = frac * lambda_total * p_mean
    return bordes, counts


def _histograma_barras(barras_crit, v_dict, lambda_por_barra, tipo, n_bins=N_BINS):
    """
    Histograma de FC/año en barras por bin de tensión.
    Usa P_FC evaluada en el valor exacto de V_mk (punto, no bin).
    """
    bordes = np.linspace(0.0, 1.0, n_bins + 1)
    counts = np.zeros(n_bins)
    ancho  = bordes[1] - bordes[0]
    for k in barras_crit:
        v = v_dict[k]
        if not np.isfinite(v):
            continue
        bi = min(int(v / ancho), n_bins - 1)
        counts[bi] += lambda_por_barra * _pfc(v, tipo)
    return bordes, counts


def graficar_perfil_y_fdp(m_nombre, curvas, v_umbral):
    """
    Para cada corredor crítico genera una figura con 2×2 subplots:
      - Fila superior: falla 3φ
      - Fila inferior: falla 1φ
      - Col izquierda: V vs ξ
      - Col derecha:   fdp f(V)
    Replica la Figura 4.5 de la memoria.
    """
    labels_vistos = set()
    criticas = []
    for c in curvas:
        if c['label'] not in labels_vistos and (c['critico_3f'] or c['critico_1f']):
            criticas.append(c)
            labels_vistos.add(c['label'])

    for c in criticas:
        fig, axes = plt.subplots(2, 2, figsize=(6.5, 5.0))
        partes = c['label'].split(' — ')
        titulo = (f"{partes[0].split()[0]}–{partes[1].split()[0]}"
                  if len(partes) == 2 else c['label'])
        fig.suptitle(f'{m_nombre}  |  Tramo {titulo}  (L={c["L_km"]:.0f} km)',
                     fontsize=12, fontweight='bold')

        for row, (tipo, pts, color) in enumerate([
            ('3φ', c['pts_3f'], COLOR_3F_LINEA),
            ('1φ', c['pts_1f'], COLOR_1F_LINEA),
        ]):
            xi_arr = np.array([p[0] for p in pts])
            V_arr  = np.array([p[1] for p in pts])
            # filtrar inf (puntos no críticos)
            mask   = np.isfinite(V_arr)
            xi_arr, V_arr = xi_arr[mask], V_arr[mask]
            if len(V_arr) == 0:
                continue
            V_fdp, fdp = _calcular_fdp(pts)

            # Col izquierda: V vs ξ
            ax = axes[row][0]
            ax.plot(xi_arr, V_arr, color=color, linewidth=1.8)
            ax.axhline(v_umbral, color=COLOR_UMBRAL, linestyle='--',
                       linewidth=1, alpha=0.7)
            ax.fill_between(xi_arr, V_arr, 0,
                            where=V_arr <= v_umbral,
                            alpha=0.12, color=color)
            ax.set_xlabel('ξ (posición de falla)', fontsize=10)
            ax.set_ylabel(f'Tensión en {m_nombre.split()[0]} (p.u.)', fontsize=10)
            ax.set_xlim(0, 1)
            ax.set_ylim(0, max(V_arr.max() * 1.05, v_umbral * 1.1))
            ax.set_title(f'Falla {tipo}', fontsize=10)
            ax.grid(True, alpha=0.25)

            # Col derecha: fdp f(V)
            ax = axes[row][1]
            mask = np.isfinite(fdp)
            ax.plot(V_fdp[mask], fdp[mask], color=color, linewidth=1.8)
            ax.axvline(v_umbral, color=COLOR_UMBRAL, linestyle='--',
                       linewidth=1, alpha=0.7)
            ax.set_xlabel(f'Tensión en {m_nombre.split()[0]} (p.u.)', fontsize=10)
            ax.set_ylabel('Densidad de probabilidad', fontsize=10)
            ax.set_xlim(0, 1)
            ax.set_ylim(bottom=0)
            ax.set_title(f'fdp tensión ({tipo})', fontsize=10)
            ax.grid(True, alpha=0.25)

        plt.tight_layout()
        nombre = f"{m_nombre.replace(' ','_')}_fig45_{titulo.replace('–','-')}.png"
        _mostrar(fig, nombre)


# ===========================================================================
# FIGURA 4.6 — Histograma fallas/año por bin de tensión
# ===========================================================================

def graficar_histograma_fallas(m_nombre, curvas, v_umbral):
    """
    Para cada corredor crítico genera un histograma de fallas/año por bin
    de tensión, con 3φ (rojo) y 1φ (azul) superpuestos.
    Replica la Figura 4.6 de la memoria.
    """
    labels_vistos = set()
    criticas = []
    for c in curvas:
        if c['label'] not in labels_vistos and (c['critico_3f'] or c['critico_1f']):
            criticas.append(c)
            labels_vistos.add(c['label'])

    for c in criticas:
        L    = c['L_km']
        lam3 = F_3F * LAMBDA_LINEA * L / 100.0
        lam1 = F_1F * LAMBDA_LINEA * L / 100.0

        bordes3, counts3 = _histograma_fallas_raw(c['pts_3f'], lam3)
        bordes1, counts1 = _histograma_fallas_raw(c['pts_1f'], lam1)

        ancho   = bordes3[1] - bordes3[0]
        centros = 0.5 * (bordes3[:-1] + bordes3[1:])
        w       = ancho * 0.44   # mitad del ancho para barras adyacentes

        fig, ax = plt.subplots(figsize=(6.5, 3.5))

        ax.bar(centros - w/2, counts1, width=w,
               color=COLOR_1F_LINEA, alpha=0.85, label=f'1φ  (λ={lam1:.4f} f/año)')
        ax.bar(centros + w/2, counts3, width=w,
               color=COLOR_3F_LINEA, alpha=0.85, label=f'3φ  (λ={lam3:.4f} f/año)')

        ax.axvline(v_umbral, color=COLOR_UMBRAL, linestyle='--',
                   linewidth=1.2, alpha=0.8, label=f'Umbral {v_umbral} p.u.')

        ax.set_xlabel('Magnitud de tensión (p.u.)', fontsize=11)
        ax.set_ylabel('Hundimientos de tensión (por año)', fontsize=11)

        partes = c['label'].split(' — ')
        titulo = (f"{partes[0].split()[0]}–{partes[1].split()[0]}"
                  if len(partes) == 2 else c['label'])
        ax.set_title(f'{m_nombre}  |  Tramo {titulo}  (L={L:.0f} km)',
                     fontsize=11, fontweight='bold')
        ax.set_xticks(bordes3[::2])
        ax.set_xticklabels([f'{v:.2f}' for v in bordes3[::2]], fontsize=7, rotation=45, ha='right')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.25, axis='y')
        ax.set_xlim(0, 1)
        ax.set_ylim(bottom=0)

        plt.tight_layout()
        nombre = f"{m_nombre.replace(' ','_')}_fig46_{titulo.replace('–','-')}.png"
        _mostrar(fig, nombre)


def graficar_frecuencia_acumulada_total(m_nombre, curvas, areas_b,
                                        v3f_barras, v1f_barras, v_umbral):
    """
    Histograma de fallas/año por bin de tensión, sumando todos los corredores
    críticos más las barras críticas (apiladas encima).
    Colores: azul=líneas 1φ, rojo=líneas 3φ, verde=barras 1φ, naranja=barras 3φ.
    """
    labels_vistos = set()
    criticas = []
    for c in curvas:
        if c['label'] not in labels_vistos and (c['critico_3f'] or c['critico_1f']):
            criticas.append(c)
            labels_vistos.add(c['label'])

    bordes   = np.linspace(0.0, 1.0, N_BINS + 1)
    # Líneas
    lin_3f = np.zeros(N_BINS)
    lin_1f = np.zeros(N_BINS)
    for c in criticas:
        L    = c['L_km']
        _, counts3 = _histograma_fallas_raw(c['pts_3f'], F_3F * LAMBDA_LINEA * L / 100.0)
        _, counts1 = _histograma_fallas_raw(c['pts_1f'], F_1F * LAMBDA_LINEA * L / 100.0)
        lin_3f += counts3
        lin_1f += counts1

    # Barras
    lam_b3 = F_3F * LAMBDA_BARRA
    lam_b1 = F_1F * LAMBDA_BARRA
    _, bar_3f = _histograma_barras_raw(areas_b['3f'], v3f_barras, lam_b3)
    _, bar_1f = _histograma_barras_raw(areas_b['1f'], v1f_barras, lam_b1)

    ancho   = bordes[1] - bordes[0]
    centros = 0.5 * (bordes[:-1] + bordes[1:])
    w       = ancho * 0.44

    fig, ax = plt.subplots(figsize=(6.5, 3.8))

    # 1φ izquierda, 3φ derecha — adyacentes
    ax.bar(centros - w/2, lin_1f, width=w,
           color=COLOR_1F_LINEA, alpha=0.85,
           label=f'Líneas 1φ  (F={F_1F:.0%})')
    ax.bar(centros + w/2, lin_3f, width=w,
           color=COLOR_3F_LINEA, alpha=0.85,
           label=f'Líneas 3φ  (F={F_3F:.0%})')

    # Barras apiladas encima de líneas
    ax.bar(centros - w/2, bar_1f, width=w,
           bottom=lin_1f,
           color=COLOR_1F_BARRA, alpha=0.85,
           label=f'Barras 1φ')
    ax.bar(centros + w/2, bar_3f, width=w,
           bottom=lin_3f,
           color=COLOR_3F_BARRA, alpha=0.85,
           label=f'Barras 3φ')

    ax.axvline(v_umbral, color=COLOR_UMBRAL, linestyle='--',
               linewidth=1.2, alpha=0.8, label=f'Umbral {v_umbral} p.u.')

    # Anotar total bajo el umbral
    idx_umb = int(round(v_umbral * N_BINS))
    total   = lin_1f + lin_3f + bar_1f + bar_3f
    f_umb   = total[:idx_umb].sum()
    ax.annotate(
        f'V < {v_umbral}: {f_umb:.4f} f/año',
        xy=(v_umbral, total[:idx_umb].max()), xycoords='data',
        xytext=(0.06, 0.80), textcoords='axes fraction',
        arrowprops=dict(arrowstyle='->', color='gray'),
        fontsize=8, color='#2c3e50',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor='gray', alpha=0.8)
    )

    ax.set_xlabel('Magnitud de tensión (p.u.)', fontsize=11)
    ax.set_ylabel('Hundimientos de tensión (por año)', fontsize=11)
    ax.set_title(
        f'{m_nombre} — Frecuencia total (líneas + barras)\n'
        f'({len(criticas)} corredores, {len(areas_b["1f"])} barras críticas 1φ, '
        f'{len(areas_b["3f"])} barras críticas 3φ)',
        fontsize=11, fontweight='bold'
    )
    ax.set_xticks(bordes[::2])
    ax.set_xticklabels([f'{v:.2f}' for v in bordes[::2]], fontsize=7, rotation=45, ha='right')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25, axis='y')
    ax.set_xlim(0, 1)
    ax.set_ylim(bottom=0)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.18)  # margen superior para que la anotación no choque con el título
    plt.tight_layout()
    nombre = f"{m_nombre.replace(' ','_')}_frecuencia_total.png"
    _mostrar(fig, nombre)


def graficar_cdf_total(m_nombre, curvas, areas_b,
                       v3f_barras, v1f_barras, v_umbral):
    """
    Gráfico de barras acumulado F(V) = fallas/año con tensión ≤ V.
    Incluye líneas y barras apiladas, con suma acumulativa de izquierda a derecha.
    """
    labels_vistos = set()
    criticas = []
    for c in curvas:
        if c['label'] not in labels_vistos and (c['critico_3f'] or c['critico_1f']):
            criticas.append(c)
            labels_vistos.add(c['label'])

    bordes = np.linspace(0.0, 1.0, N_BINS + 1)
    lin_3f = np.zeros(N_BINS)
    lin_1f = np.zeros(N_BINS)

    for c in criticas:
        L = c['L_km']
        _, counts3 = _histograma_fallas_raw(c['pts_3f'], F_3F * LAMBDA_LINEA * L / 100.0)
        _, counts1 = _histograma_fallas_raw(c['pts_1f'], F_1F * LAMBDA_LINEA * L / 100.0)
        lin_3f += counts3
        lin_1f += counts1

    lam_b3 = F_3F * LAMBDA_BARRA
    lam_b1 = F_1F * LAMBDA_BARRA
    _, bar_3f = _histograma_barras_raw(areas_b['3f'], v3f_barras, lam_b3)
    _, bar_1f = _histograma_barras_raw(areas_b['1f'], v1f_barras, lam_b1)

    # Suma acumulativa — todos los bins
    cdf_lin_1f = np.cumsum(lin_1f)
    cdf_lin_3f = np.cumsum(lin_3f)
    cdf_bar_1f = np.cumsum(bar_1f)
    cdf_bar_3f = np.cumsum(bar_3f)
    cdf_total  = cdf_lin_1f + cdf_lin_3f + cdf_bar_1f + cdf_bar_3f

    ancho   = bordes[1] - bordes[0]
    centros = 0.5 * (bordes[:-1] + bordes[1:])
    bordes_plot = bordes

    fig, ax = plt.subplots(figsize=(6.5, 3.8))

    # Líneas (base)
    ax.bar(centros, cdf_lin_1f, width=ancho * 0.95,
           color=COLOR_1F_LINEA, alpha=0.85, label=f'Líneas 1φ  (F={F_1F:.0%})')
    ax.bar(centros, cdf_lin_3f, width=ancho * 0.95,
           color=COLOR_3F_LINEA, alpha=0.75, label=f'Líneas 3φ  (F={F_3F:.0%})')

    # Barras (apiladas)
    ax.bar(centros, cdf_bar_1f, width=ancho * 0.95,
           bottom=cdf_lin_1f,
           color=COLOR_1F_BARRA, alpha=0.85, label=f'Barras 1φ  (F={F_1F:.0%})')
    ax.bar(centros, cdf_bar_3f, width=ancho * 0.95,
           bottom=cdf_lin_3f,
           color=COLOR_3F_BARRA, alpha=0.75, label=f'Barras 3φ  (F={F_3F:.0%})')

    ax.axvline(v_umbral, color=COLOR_UMBRAL, linestyle='--',
               linewidth=1.2, alpha=0.8, label=f'Umbral {v_umbral} p.u.')

    # Anotar valor en el umbral
    idx_umb = int(round(v_umbral / ancho)) - 1
    idx_umb = min(idx_umb, N_BINS - 1)
    f_umb   = cdf_total[idx_umb]
    ax.annotate(
        f'F({v_umbral}) = {f_umb:.4f} f/año',
        xy=(centros[idx_umb], cdf_total[idx_umb]), xycoords='data',
        xytext=(0.06, 0.80), textcoords='axes fraction',
        arrowprops=dict(arrowstyle='->', color='gray'),
        fontsize=9, color='#2c3e50',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor='gray', alpha=0.8)
    )

    ax.set_xlabel('Magnitud de tensión V (p.u.)', fontsize=11)
    ax.set_ylabel('Hundimientos acumulados (por año)', fontsize=11)
    ax.set_title(
        f'{m_nombre} — F(V): fallas/año con tensión ≤ V\n'
        f'(líneas + barras)',
        fontsize=12, fontweight='bold'
    )
    ax.set_xticks(bordes_plot[::2])
    ax.set_xticklabels([f'{v:.2f}' for v in bordes_plot[::2]], fontsize=7, rotation=45, ha='right')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25, axis='y')
    ax.set_xlim(0, bordes_plot[-1])
    ax.set_ylim(bottom=0)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.18)  # margen superior para que la anotación no choque con el título
    plt.tight_layout()
    nombre = f"{m_nombre.replace(' ','_')}_cdf_total.png"
    _mostrar(fig, nombre)

def graficar_fc_acumulada(m_nombre, curvas, areas_b,
                          v3f_barras, v1f_barras, v_umbral):
    """
    Histograma acumulado de FC/año (ponderado por P_FC).
    Gráfico final por barra de observación.
    """
    labels_vistos = set()
    criticas = []
    for c in curvas:
        if c['label'] not in labels_vistos and (c['critico_3f'] or c['critico_1f']):
            criticas.append(c)
            labels_vistos.add(c['label'])

    bordes = np.linspace(0.0, 1.0, N_BINS + 1)
    lin_3f = np.zeros(N_BINS)
    lin_1f = np.zeros(N_BINS)

    for c in criticas:
        L = c['L_km']
        _, cnt3 = _histograma_fallas(c['pts_3f'], F_3F * LAMBDA_LINEA * L / 100.0, '3f')
        _, cnt1 = _histograma_fallas(c['pts_1f'], F_1F * LAMBDA_LINEA * L / 100.0, '1f')
        lin_3f += cnt3
        lin_1f += cnt1

    _, bar_3f = _histograma_barras(areas_b['3f'], v3f_barras, F_3F * LAMBDA_BARRA, '3f')
    _, bar_1f = _histograma_barras(areas_b['1f'], v1f_barras, F_1F * LAMBDA_BARRA, '1f')

    cdf_lin_1f = np.cumsum(lin_1f)
    cdf_lin_3f = np.cumsum(lin_3f)
    cdf_bar_1f = np.cumsum(bar_1f)
    cdf_bar_3f = np.cumsum(bar_3f)

    # No dibujar datos en los bins con V >= v_umbral, pero mantenerlos visibles
    idx_desde_umbral = int(round(v_umbral * N_BINS))
    for arr in [cdf_lin_1f, cdf_lin_3f, cdf_bar_1f, cdf_bar_3f]:
        arr[idx_desde_umbral:] = 0

    ancho   = bordes[1] - bordes[0]
    centros = 0.5 * (bordes[:-1] + bordes[1:])

    fig, ax = plt.subplots(figsize=(6.5, 3.8))

    # Apilado: líneas 1φ (base) → barras 1φ → líneas 3φ → barras 3φ
    ax.bar(centros, cdf_lin_1f, width=ancho * 0.95,
           color=COLOR_1F_LINEA, alpha=0.85, label=f'Líneas 1φ')
    ax.bar(centros, cdf_bar_1f, width=ancho * 0.95,
           bottom=cdf_lin_1f,
           color=COLOR_1F_BARRA, alpha=0.85, label=f'Barras 1φ')
    ax.bar(centros, cdf_lin_3f, width=ancho * 0.95,
           bottom=cdf_lin_1f + cdf_bar_1f,
           color=COLOR_3F_LINEA, alpha=0.85, label=f'Líneas 3φ')
    ax.bar(centros, cdf_bar_3f, width=ancho * 0.95,
           bottom=cdf_lin_1f + cdf_bar_1f + cdf_lin_3f,
           color=COLOR_3F_BARRA, alpha=0.85, label=f'Barras 3φ')
    ax.axvline(v_umbral, color=COLOR_UMBRAL, linestyle='--',
               linewidth=1.2, alpha=0.8, label=f'Umbral {v_umbral} p.u.')

    # Anotar tasa total de FC en el umbral (bin inmediatamente anterior a v_umbral)
    idx_umb = int(round(v_umbral * N_BINS)) - 1
    idx_umb = max(0, min(idx_umb, N_BINS - 1))
    f_fc    = (cdf_lin_1f + cdf_bar_1f + cdf_lin_3f + cdf_bar_3f)[idx_umb]
    f_total_plot = cdf_lin_1f + cdf_bar_1f + cdf_lin_3f + cdf_bar_3f
    ax.annotate(
        f'FC/año = {f_fc:.4f}',
        xy=(centros[idx_umb], f_total_plot[idx_umb]), xycoords='data',
        xytext=(0.06, 0.80), textcoords='axes fraction',
        arrowprops=dict(arrowstyle='->', color='gray'),
        fontsize=9, color='#2c3e50', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                  edgecolor='gray', alpha=0.8)
    )

    ax.set_xlabel('Magnitud de tensión V (p.u.)', fontsize=11)
    ax.set_ylabel('FC acumuladas [FC/año]', fontsize=11)
    ax.set_title(
        f'{m_nombre} — Tasa anual de fallas de conmutación\n'
        f'({len(criticas)} corredores + {len(areas_b["1f"])} barras críticas)',
        fontsize=11, fontweight='bold'
    )
    ax.set_xticks(bordes[::2])
    ax.set_xticklabels([f'{v:.2f}' for v in bordes[::2]], fontsize=7, rotation=45, ha='right')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25, axis='y')
    ax.set_xlim(0, 1)
    ax.set_ylim(bottom=0)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.18)  # margen superior para que la anotación no choque con el título
    plt.tight_layout()
    nombre = f"{m_nombre.replace(' ','_')}_fc_acumulada.png"
    _mostrar(fig, nombre)



def calcular_tasa_fc(curvas, areas_b, v3f_barras, v1f_barras, v_umbral):
    """
    Calcula la tasa anual de FC (líneas + barras, 3φ + 1φ) para una barra
    de observación. Replica exactamente la integración usada en
    graficar_fc_acumulada, de modo que el valor coincida con el anotado
    en la figura.

    Retorna un dict con el desglose por origen/tipo de falla y el total.
    """
    labels_vistos = set()
    criticas = []
    for c in curvas:
        if c['label'] not in labels_vistos and (c['critico_3f'] or c['critico_1f']):
            criticas.append(c)
            labels_vistos.add(c['label'])

    lin_3f = np.zeros(N_BINS)
    lin_1f = np.zeros(N_BINS)
    for c in criticas:
        L = c['L_km']
        _, cnt3 = _histograma_fallas(c['pts_3f'], F_3F * LAMBDA_LINEA * L / 100.0, '3f')
        _, cnt1 = _histograma_fallas(c['pts_1f'], F_1F * LAMBDA_LINEA * L / 100.0, '1f')
        lin_3f += cnt3
        lin_1f += cnt1

    _, bar_3f = _histograma_barras(areas_b['3f'], v3f_barras, F_3F * LAMBDA_BARRA, '3f')
    _, bar_1f = _histograma_barras(areas_b['1f'], v1f_barras, F_1F * LAMBDA_BARRA, '1f')

    cdf_lin_3f = np.cumsum(lin_3f)
    cdf_lin_1f = np.cumsum(lin_1f)
    cdf_bar_3f = np.cumsum(bar_3f)
    cdf_bar_1f = np.cumsum(bar_1f)

    idx_umb = int(round(v_umbral * N_BINS)) - 1
    idx_umb = max(0, min(idx_umb, N_BINS - 1))

    return dict(
        lineas_3f=cdf_lin_3f[idx_umb],
        lineas_1f=cdf_lin_1f[idx_umb],
        barras_3f=cdf_bar_3f[idx_umb],
        barras_1f=cdf_bar_1f[idx_umb],
        total=(cdf_lin_3f + cdf_lin_1f + cdf_bar_3f + cdf_bar_1f)[idx_umb],
    )


def imprimir_tasas_fc(curvas_obs, areas_b, v3f_barras, v1f_barras,
                       obs_buses, v_umbral):
    """
    Resumen en texto de la tasa anual de fallas de conmutación (FC/año)
    por barra de observación, con desglose por origen (líneas/barras) y
    tipo de falla (3φ/1φ). Usa la misma integración que graficar_fc_acumulada,
    por lo que el total coincide con la anotación de esa figura.
    """
    ANCHO = 60
    print(f"\n{'='*ANCHO}")
    print("  TASA ANUAL DE FALLAS DE CONMUTACIÓN (FC/año)")
    print('='*ANCHO)
    for m, m_nombre in obs_buses.items():
        r = calcular_tasa_fc(curvas_obs[m], areas_b[m],
                              v3f_barras[m], v1f_barras[m], v_umbral)
        print(f"\n  ► {m_nombre}  (barra {m})")
        print(f"    Líneas   3φ: {r['lineas_3f']:.4f}   1φ: {r['lineas_1f']:.4f}")
        print(f"    Barras   3φ: {r['barras_3f']:.4f}   1φ: {r['barras_1f']:.4f}")
        print(f"    {'-'*38}")
        print(f"    TOTAL FC/año: {r['total']:.4f}")
    print(f"\n{'='*ANCHO}")


def generar_graficos(curvas_obs, areas_b, areas_l,
                     v3f_barras, v1f_barras,
                     obs_buses, nombres, v_umbral):
    """
    Genera Figuras 4.5 y 4.6 de la memoria para cada barra de observación,
    más un gráfico final de frecuencia acumulada total por barra.
    """
    for m, m_nombre in obs_buses.items():
        curvas = curvas_obs[m]
        n_crit = len([c for c in curvas if c['critico_3f'] or c['critico_1f']])
        print(f"\n{'='*65}")
        print(f"  GRÁFICOS — {m_nombre}  ({n_crit} corredores críticos)")
        print(f"{'='*65}")

        print(f"  → Figura 4.5: perfil V(ξ) y fdp por corredor")
        graficar_perfil_y_fdp(m_nombre, curvas, v_umbral)

        print(f"  → Figura 4.6: histograma fallas/año por corredor")
        graficar_histograma_fallas(m_nombre, curvas, v_umbral)

        print(f"  → Frecuencia acumulada total")
        graficar_frecuencia_acumulada_total(
            m_nombre, curvas, areas_b[m],
            v3f_barras[m], v1f_barras[m], v_umbral
        )

        print(f"  → CDF: hundimientos acumulados")
        graficar_cdf_total(
            m_nombre, curvas, areas_b[m],
            v3f_barras[m], v1f_barras[m], v_umbral
        )

        print(f"  → FC acumuladas: tasa anual de FC")
        graficar_fc_acumulada(
            m_nombre, curvas, areas_b[m],
            v3f_barras[m], v1f_barras[m], v_umbral
        )


if __name__ == '__main__':
    print("Importar desde main_sic.py para generar gráficos.")


## Módulo 6 — `main_sic.py`

In [ ]:
%%writefile main_sic.py
"""
main_sic.py — Análisis Completo STN 25 Barras (v1.0)
======================================================
Metodología: VR (2026)

Cambios respecto a v1.0:
  - Z_barra de secuencia cero construida desde DATA0. En v1.0 la matriz
    homopolar era una copia de la de secuencia positiva: DATA0 se
    importaba pero nunca se usaba en el flujo principal.
  - Filtro de cobertura en el cálculo monofásico sobre líneas: solo se
    evalúan corredores con Z0_mk simulado en ambos extremos. Sin este
    filtro se interpolaba entre un extremo calibrado y otro algorítmico,
    con escalas separadas por dos órdenes de magnitud, lo que producía
    mínimos interiores de ~0 p.u. en corredores cuyos dos extremos
    superaban 0,94 p.u.
  - El clamp de tensión a [0,1] emite aviso en vez de operar en silencio.
  - Corregido el nombre de la barra 6: Punta Colorada.

  NOTA. No se modifica la asignación Z0_kj = Z1_kj del término cruzado de
  Ec. 3.16. Es dimensionalmente incorrecta, pero no existe dato calibrado
  de Z0_kj y sustituirla por el valor algorítmico introduce un error mayor
  (las tasas caían de 4,40/5,88/4,39 a 3,08/1,10/1,09 FC/año). La solución
  definitiva es calibrarla con una simulación monofásica por corredor:
      Z0_kj = (1 - V_a,kj_sim)·(2·Z1_jj + Z0_jj) - 2·Z1_kj
  análoga a la Ec. 3.41 ya empleada en secuencia positiva.

Cambios de v1.0 a v1.0:
  - Elimina el parche Z1_eff_1f (calibrar_zmk_1f_simulado), que era
    metodológicamente inconsistente: asignaba dos valores distintos de
    Z1_mk según el tipo de falla, sin justificación física.
  - Incorpora Z_MK_0 (sic_datos.py): impedancias de transferencia de
    secuencia cero simuladas directamente en PowerFactory.
  - tension_falla_barra_1f y tension_falla_linea_1f usan la fórmula
    exacta de tres secuencias:
        V_a_mk = 1 - (2·Z1_mk + Z0_mk) / (2·Z1_kk + Z0_kk)
    con Z1_mk idéntico al de la rama 3φ (consistencia física) y Z0_mk
    de simulación directa para los pares disponibles.

Flujo:
  1. Z_barra base desde algoritmo (Apéndice B), secuencias positiva y cero
  2. Calibración con datos de simulación DIgSILENT:
       - Diagonal Z_kk: exacta desde Scc3 y Scc1 (todas las barras)
       - Off-diagonal Z1_mk: exacta desde V_sim_3f (Ec. 3.8 inversa)
       - Off-diagonal Z0_mk: exacta desde Z_MK_0 (NUEVO v1.0)
  3. Tensiones de falla en barras:  fórmula 3-secuencias con Z_MK_0
  4. Tensiones de falla en líneas:  ídem con interpolación de Z0_mp
  5. Áreas de vulnerabilidad
"""
import sys, os
import numpy as np
sys.path.insert(0, os.path.dirname(__file__))

from sic_datos import (
    DATA1, DATA0, NBUS, NOMBRES, BARRAS_OBS, SBASE, V_UMBRAL, V_NO_CRITICO,
    get_corredores, get_zkk_simulacion, get_zkj_simulacion,
    SCC3_MVA, SCC1_MVA, V_SIM_3F, V_SIM_1F, V_KJ,
    Z_MK_1, Z_MK_0,
)
from zbarra import construir_zbarra, imprimir_zbarra, calibrar_zbarra_sim
from cortocircuito import calcular_potencias_cc, imprimir_tabla_cc
from tensiones_falla import (
    tension_falla_barra_3f, tension_falla_barra_1f,
    tension_falla_linea_3f, tension_falla_linea_1f,
    areas_vulnerabilidad_barras, areas_vulnerabilidad_lineas,
    curvas_area_vulnerabilidad,
    imprimir_tensiones_barras, imprimir_tensiones_lineas,
    imprimir_areas_vulnerabilidad,
)
from graficos import generar_graficos, imprimir_tasas_fc


def run_analisis():
    """
    Ejecuta el análisis completo y retorna todos los resultados.
    Llamar desde notebook para luego generar gráficos inline.
    """
    print("\n" + "="*65)
    print("  ANALISIS DE FALLAS — STN SIMPLIFICADO 25 BARRAS  (v1.0)")
    print("  Metodologia: VR (2026) + Z_MK_0 simulado")
    print("="*65)
    print(f"  Sbase={SBASE:.0f} MVA | Barras:{NBUS} | Umbral V={V_UMBRAL} p.u.")

    print("\n[1/5] Construyendo Z_barra base (algoritmo Apendice B) ...")
    Z_algo   = construir_zbarra(DATA1, NBUS)   # secuencia positiva
    Z_algo_0 = construir_zbarra(DATA0, NBUS)   # secuencia cero (v1.0)

    print("[2/5] Calibrando Z_barra con datos de simulacion ...")
    Z1kk_sim, Z0kk_sim = get_zkk_simulacion()
    zkj_sim = get_zkj_simulacion(Z1kk_sim)
    Z1, Z0  = calibrar_zbarra_sim(Z_algo, Z1kk_sim, Z0kk_sim, V_SIM_3F,
                                  obs_buses=list(BARRAS_OBS.keys()),
                                  v_default=V_NO_CRITICO,
                                  Z_algo_0=Z_algo_0)
    print(f"  Diagonal Z_kk:        exacta desde Scc3/Scc1 (25 barras)")
    print(f"  Off-diagonal Z1_mk:   exacta desde V_sim_3f ({len(V_SIM_3F)} pares)")
    print(f"  Off-diagonal Z0_mk:   exacta desde Z_MK_0 ({len(Z_MK_0)} pares)")
    print(f"  Z_kj extremos linea:  exacta desde V_kj ({len(V_KJ)} lineas)")
    print(f"  Z_barra sec. cero:    algoritmo sobre DATA0 (v1.0)")
    print(f"  Filtro cobertura 1f:  solo corredores con Z0_mk simulado en "
          f"ambos extremos")

    print("\n[3/5] Verificando potencias de cortocircuito ...")
    res_cc = calcular_potencias_cc(Z1, Z0, Sbase=SBASE, nombres=NOMBRES)
    imprimir_tabla_cc(res_cc, Sbase=SBASE)
    print(f"\n  Verificacion barras de observacion:")
    for barra, nombre in BARRAS_OBS.items():
        r = res_cc[barra - 1]
        print(f"    {nombre}: Scc3={r['Scc3_MVA']:.1f} MVA "
              f"(sim={SCC3_MVA[barra-1]:.1f})  |  "
              f"Scc1={r['Scc1_MVA']:.1f} MVA "
              f"(sim={SCC1_MVA[barra-1]:.1f})")

    print("\n[4/5] Calculando tensiones de falla ...")
    corredores = get_corredores()
    v3f  = tension_falla_barra_3f(Z1, BARRAS_OBS, V_SIM_3F)
    v1f  = tension_falla_barra_1f(Z1, Z0, BARRAS_OBS,
                                   z_mk_1=Z_MK_1, z_mk_0=Z_MK_0,
                                   v_sim_1f=V_SIM_1F, v_sim_3f=V_SIM_3F)
    vl3f = tension_falla_linea_3f(Z1, corredores, BARRAS_OBS,
                                   v_sim=V_SIM_3F, zkj_sim=zkj_sim,
                                   v_umbral=V_UMBRAL)
    vl1f = tension_falla_linea_1f(Z1, Z0, corredores, BARRAS_OBS,
                                   v_sim_3f=V_SIM_3F, v_sim_1f=V_SIM_1F,
                                   zkj_sim=zkj_sim, v_umbral=V_UMBRAL,
                                   z_mk_1=Z_MK_1, z_mk_0=Z_MK_0)
    imprimir_tensiones_barras(v3f, v1f, BARRAS_OBS, NOMBRES, V_UMBRAL)
    imprimir_tensiones_lineas(vl3f, vl1f, BARRAS_OBS, V_UMBRAL)

    print("\n[5/5] Identificando areas de vulnerabilidad ...")
    areas_b = areas_vulnerabilidad_barras(v3f, v1f, BARRAS_OBS, NOMBRES, V_UMBRAL)
    areas_l = areas_vulnerabilidad_lineas(vl3f, vl1f, BARRAS_OBS, V_UMBRAL)
    imprimir_areas_vulnerabilidad(areas_b, areas_l, BARRAS_OBS, NOMBRES)

    print(f"\n{'='*65}")
    print("  RESUMEN EJECUTIVO")
    print(f"{'='*65}")
    for m, nombre in BARRAS_OBS.items():
        b3 = areas_b[m]['3f'];  b1 = areas_b[m]['1f']
        l3 = areas_l[m]['3f'];  l1 = areas_l[m]['1f']
        print(f"\n  {nombre}:")
        print(f"    Barras criticas -> 3f: {len(b3)}  |  1f: {len(b1)}")
        print(f"    Lineas criticas -> 3f: {len(l3)}  |  1f: {len(l1)}")
    print(f"{'='*65}\n")

    curvas_obs = curvas_area_vulnerabilidad(
        vl3f, vl1f, areas_l, corredores, BARRAS_OBS
    )

    imprimir_tasas_fc(curvas_obs, areas_b, v3f, v1f, BARRAS_OBS, V_UMBRAL)

    return dict(
        Z1=Z1, Z0=Z0,
        v3f=v3f, v1f=v1f,
        vl3f=vl3f, vl1f=vl1f,
        areas_b=areas_b, areas_l=areas_l,
        curvas_obs=curvas_obs,
        corredores=corredores,
    )


def main():
    """Punto de entrada CLI — analisis + graficos."""
    res = run_analisis()
    print("\n[6/6] Generando graficos ...")
    generar_graficos(
        res['curvas_obs'], res['areas_b'], res['areas_l'],
        res['v3f'], res['v1f'],
        BARRAS_OBS, NOMBRES, V_UMBRAL
    )


if __name__ == '__main__':
    main()


## ▶ Análisis numérico

In [ ]:
import sys
sys.path.insert(0, '.')
from main_sic import run_analisis

res = run_analisis()


In [ ]:
# Exportación de los CSV que alimentan las figuras del Capítulo 4
import os
import numpy as np
import pandas as pd
from graficos import (_histograma_fallas_raw, _histograma_fallas,
                      _histograma_barras_raw, _histograma_barras,
                      N_BINS, F_3F, F_1F, LAMBDA_LINEA, LAMBDA_BARRA)
from sic_datos import BARRAS_OBS, V_UMBRAL

RUTA_DATOS = 'datos_tex'
os.makedirs(RUTA_DATOS, exist_ok=True)


def exportar_datos_histogramas(m_nombre, curvas, areas_b, v3f_barras, v1f_barras,
                                v_umbral, ruta_dir=RUTA_DATOS):
    labels_vistos = set()
    criticas = []
    for c in curvas:
        if c['label'] not in labels_vistos and (c['critico_3f'] or c['critico_1f']):
            criticas.append(c)
            labels_vistos.add(c['label'])

    bordes = np.linspace(0.0, 1.0, N_BINS + 1)
    lin_3f = np.zeros(N_BINS); lin_1f = np.zeros(N_BINS)
    lin_3f_fc = np.zeros(N_BINS); lin_1f_fc = np.zeros(N_BINS)

    for c in criticas:
        L = c['L_km']
        _, c3  = _histograma_fallas_raw(c['pts_3f'], F_3F * LAMBDA_LINEA * L / 100.0)
        _, c1  = _histograma_fallas_raw(c['pts_1f'], F_1F * LAMBDA_LINEA * L / 100.0)
        lin_3f += c3; lin_1f += c1
        _, c3f = _histograma_fallas(c['pts_3f'], F_3F * LAMBDA_LINEA * L / 100.0, '3f')
        _, c1f = _histograma_fallas(c['pts_1f'], F_1F * LAMBDA_LINEA * L / 100.0, '1f')
        lin_3f_fc += c3f; lin_1f_fc += c1f

    lam_b3, lam_b1 = F_3F * LAMBDA_BARRA, F_1F * LAMBDA_BARRA
    _, bar_3f = _histograma_barras_raw(areas_b['3f'], v3f_barras, lam_b3)
    _, bar_1f = _histograma_barras_raw(areas_b['1f'], v1f_barras, lam_b1)
    _, bar_3f_fc = _histograma_barras(areas_b['3f'], v3f_barras, F_3F * LAMBDA_BARRA, '3f')
    _, bar_1f_fc = _histograma_barras(areas_b['1f'], v1f_barras, F_1F * LAMBDA_BARRA, '1f')

    # --- Fig. 4.9: acumulado SIN ponderar por P_FC ---
    cdf_lin_1f = np.cumsum(lin_1f); cdf_lin_3f = np.cumsum(lin_3f)
    cdf_bar_1f = np.cumsum(bar_1f); cdf_bar_3f = np.cumsum(bar_3f)
    cdf_total  = cdf_lin_1f + cdf_lin_3f + cdf_bar_1f + cdf_bar_3f

    # --- Fig. 4.10: acumulado CON ponderación P_FC (y truncado en el umbral) ---
    cdf_lin_1f_fc = np.cumsum(lin_1f_fc); cdf_lin_3f_fc = np.cumsum(lin_3f_fc)
    cdf_bar_1f_fc = np.cumsum(bar_1f_fc); cdf_bar_3f_fc = np.cumsum(bar_3f_fc)
    idx_umbral = int(round(v_umbral * N_BINS))
    for arr in [cdf_lin_1f_fc, cdf_lin_3f_fc, cdf_bar_1f_fc, cdf_bar_3f_fc]:
        arr[idx_umbral:] = 0

    centros = 0.5 * (bordes[:-1] + bordes[1:])
    df = pd.DataFrame({
        'V_centro': centros,
        'cdf_lin_1f': cdf_lin_1f, 'cdf_lin_3f': cdf_lin_3f,
        'cdf_bar_1f': cdf_bar_1f, 'cdf_bar_3f': cdf_bar_3f,
        'cdf_lin_1f_fc': cdf_lin_1f_fc, 'cdf_lin_3f_fc': cdf_lin_3f_fc,
        'cdf_bar_1f_fc': cdf_bar_1f_fc, 'cdf_bar_3f_fc': cdf_bar_3f_fc,
    })
    df.to_csv(f'{ruta_dir}/{m_nombre.split()[0]}_histogramas.csv', index=False)

    # --- Resumen de valores a anotar (F(V) y FC/año en el umbral) ---
    idx_umb9  = min(int(round(v_umbral / (bordes[1]-bordes[0]))) - 1, N_BINS - 1)
    f_umb     = cdf_total[idx_umb9]
    idx_umb10 = max(0, min(int(round(v_umbral * N_BINS)) - 1, N_BINS - 1))
    fc_umb    = (cdf_lin_1f_fc + cdf_bar_1f_fc + cdf_lin_3f_fc + cdf_bar_3f_fc)[idx_umb10]
    pd.DataFrame([{'bus': m_nombre, 'F_umbral': f_umb, 'FC_umbral': fc_umb, 'v_umbral': v_umbral}]
                 ).to_csv(f'{ruta_dir}/{m_nombre.split()[0]}_resumen.csv', index=False)

    print(f"  [OK] {m_nombre}: F({v_umbral})={f_umb:.4f} f/año | FC/año={fc_umb:.4f}")

# Ejecutar para las tres barras de observación
for _m, _nombre in BARRAS_OBS.items():
    exportar_datos_histogramas(_nombre, res['curvas_obs'][_m], res['areas_b'][_m],
                               res['v3f'][_m], res['v1f'][_m], V_UMBRAL)


In [ ]:
import shutil
from google.colab import files

shutil.make_archive('datos_tex', 'zip', 'datos_tex')
files.download('datos_tex.zip')

## ▶ Gráficos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from graficos import generar_graficos
from sic_datos import BARRAS_OBS, NOMBRES, V_UMBRAL

generar_graficos(
    res['curvas_obs'], res['areas_b'], res['areas_l'],
    res['v3f'], res['v1f'],
    BARRAS_OBS, NOMBRES, V_UMBRAL
)
